# 02 - Ingeniería y Tranformación de Características 

## 1. Descripción General

La ingeniería de características (*feature Engineering*) es el proceso de transformar datos brutos en entradas significativas (características) que ayuden a los modelos de machine learning a aprender patrones de manera más efectiva y mejorar la precisión de las predicciones. Esto implica la creación de nuevas características, transformación de las características existentes, imputación de valores faltantes, codificación de datos categóricos y selección de las características más relevantes.

Dado que el rendimiento de un modelo depende en gran medida de la calidad de los datos de entrenamiento, la ingeniería de características se convierte en un paso crucial del preprocesamiento. El objetivo de esta sección será transformar y seleccionar los aspectos más relevantes de los datos brutos, alineándolos tanto con la tarea predictiva como con las necesidades específicas del modelo, para maximizar su capacidad de aprendizaje y generalización.

En esta etapa dejamos de “explorar” y empezamos a construir lo que el modelo realmente va a usar. Todo el EDA previo tuvo un propósito claro: identificar qué variables contienen señal y cuáles no. Aquí convertiremos ese análisis en un proceso reproducible.

Partiremos del conjunto de datos `df_features` que contiene las características candidatas al modelo y construiremos un conjunto de variables (X) que capturen las señales mas relevantes sobre los factores que influyen en la fijación de precios de los alojamientos de Airbnb en la Ciudad de México. Para lograrlo, se dividirá el proceso en 3 niveles:

- **Limpieza de características (`clean_features`):**  
Aplicar pasos iniciales de limpieza y ajuste sobre las variables candidatas. Esto puede incluir conversión de tipos de datos, tratamiento de columnas y normalización de formatos. El objetivo es garantizar que las variables estén en condiciones adecuadas para la creación y transformación posterior.

- **Creación de características (`build_features`):**  
Generar nuevas variables a partir de los datos originales. Estas pueden incluir features o métricas derivadas, combinaciones de atributos o indicadores que aporten señales adicionales al modelo.

- **Preprocesamiento y transformación de características (`preprocess_features`):**  
Transformar las variables existentes para optimizar su representación. Esto abarca operaciones como normalización, imputación, escalado, codificación de categorías o transformaciones matemáticas que mejoren la capacidad predictiva.

**Flujo completo**

```text
df_features
   ↓
df_prepared
   ↓
clean_features()
   ↓
build_features()
   |
   | → preprocess_features()
   │       ↓
   │     df_eval → Dataset para evaluar y seleccionar variables
   │
   ↓
df_model → Dataset final con las variables seleccionadas para el modelo.

## 2. Importación de Librerías y Carga del Dataset

In [41]:
# Import libraries and modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath(".."))

# Load the autoreload extension to automatically reload modules when they change
# Set autoreload mode to 2: reload all modules before executing user code
%load_ext autoreload
%autoreload 2

# Load the dataset generated during the Data Exploration phase
df_features = pd.read_csv("../data/processed/df_features.csv")

# Preview first rows
df_features.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,beds,minimum_nights,...,instant_bookable,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,clean_price,log_price
0,Cuajimalpa de Morelos,19.38283,-99.27178,Entire villa,Entire home/apt,2,1.0,1.0,1.0,1,...,f,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3673.0,8.208764
1,Cuauhtémoc,19.41162,-99.17794,Entire home,Entire home/apt,14,5.5,5.0,8.0,1,...,f,4.59,4.56,4.70,4.87,4.78,4.98,4.47,18000.0,9.798127
2,Cuauhtémoc,19.43977,-99.15605,Entire condo,Entire home/apt,2,1.0,1.0,1.0,15,...,f,4.87,4.95,4.88,4.98,4.94,4.76,4.79,591.0,6.381816
3,Miguel Hidalgo,19.40826,-99.18659,Entire home,Entire home/apt,16,5.0,5.0,10.0,2,...,f,4.88,4.91,4.84,4.91,4.89,4.75,4.90,3673.0,8.208764
4,Benito Juárez,19.39675,-99.17581,Private room in rental unit,Private room,2,1.0,1.0,1.0,4,...,f,4.84,4.86,4.61,4.98,4.95,4.97,4.83,321.0,5.771441


## 3. Limpieza y Formateo de Características (Feature Cleaning and Formatting)

Antes de iniciar el proceso de feature engineering y transformación de variables, es necesario realizar una etapa inicial de limpieza y formateo orientada a estandarizar y normalizar la estructura de los datos originales. El objetivo de esta fase es asegurar que las variables presenten formatos consistentes y tipos de datos adecuados para las siguientes etapas del pipeline. Esto permite reducir inconsistencias provenientes de los datos crudos y garantizar compatibilidad con los procesos posteriores de construcción de features, transformación estadística y modelado.

A diferencia de las transformaciones se aplicarán posteriormente a las características, esta etapa se enfoca principalmente en la limpieza y normalización básica de los datos, sin alterar todavía la distribución estadística o representacion de las variables.

Las funciones utilizadas en esta sección fueron previamente modularizadas dentro de la carpeta `src/preprocess` del proyecto, lo que permite mantener una organización clara y reutilizable. El módulo `cleaning` actuará como orquestador de la limpieza y formateo de features, integrando cada módulo en un flujo único y consistente. Este enfoque no solo mejora la calidad de las features, sino que también asegura que el proceso pueda escalar y aplicarse de forma consistente en entornos de entrenamiento e inferencia.

Esta etapa representa la primera capa del pipeline de preparación de datos y establece una base consistente para las fases posteriores de feature engineering y modelado. Se creará el dataset `df_prepared`, en el que se implentarán las etapas limpieza, creación de variables derivadas, preprocesamiento y transformaciones.

In [42]:
# Import orchestrator to clean and format feature columns
from src.preprocess.cleaning import clean_features

# Apply orchestrator → prepare dataset with cleaned and formatted features
df_prepared = clean_features(df_features)

# Display the first rows of the cleaned and formatted feature columns
df_prepared[['host_is_superhost', 'instant_bookable', 'host_verifications', 'amenities', 'room_type']]

,host_is_superhost,instant_bookable,host_verifications,amenities,room_type
0,0.0,0.0,"[email, phone, work_email]","[Garden view, Resort access, Washer, Courtyard...",entire_home/apt
1,0.0,0.0,"[email, phone, work_email]","[Piano, Patio or balcony, Wifi, Refrigerator, ...",entire_home/apt
2,1.0,0.0,"[email, phone]","[Self check-in, Dining table, Elevator, Wifi, ...",entire_home/apt
3,1.0,0.0,"[email, phone]","[Pack ’n play/Travel crib, Self check-in, Heat...",entire_home/apt
4,1.0,0.0,"[email, phone]","[Elevator, Wifi, Hot water, Microwave, Refrige...",private_room
...,...,...,...,...,...
21445,1.0,1.0,"[email, phone]","[Private entrance, Laundromat nearby, Dining t...",private_room
21446,0.0,0.0,"[email, phone]","[Washer, Iron, Kitchen, Clothing storage, Hang...",private_room
21447,0.0,1.0,"[email, phone]","[Paid parking off premises, Private entrance, ...",entire_home/apt
21448,0.0,0.0,[phone],"[Exterior security cameras on property, Kitche...",entire_home/apt


Como resultado de esta etapa, se realizaron procesos de conversión de datos, normalización y estandarización sobre variables cuyo formato original no era directamente compatible con las siguientes fases del pipeline.

En particular:

- `host_is_superhost` e `instant_bookable` fueron transformadas desde representaciones categóricas ("t" y "f") hacia valores binarios numéricos (1.0 y 0.0).
- `host_verifications` y `amenities` fueron convertidas desde cadenas con listas en formato string hacia listas de Python.
- `room_type` fue normalizada al estandarizar valores categóricos, se convirtieron a minúsculas y se reemplazaron espacios por guiones bajos ("Entire Home" → "entire_home")

Estas transformaciones permitirán asegurar consistencia en los tipos de datos y preparan el dataset para las etapas posteriores de feature engineering, transformación y modelado.

## 4. Creación de Características (Build Features)

En esta sección se construirán las variables derivadas que constituirán el conjunto de entrada (X) para el modelo. A partir de las variables originales, se generaran nuevas features con capacidad explicativa. El enfoque seguido no es arbitrario: cada feature propuesta proviene del análisis exploratorio previo, donde se identificaron patrones, relaciones y posibles fuentes de información predictiva.

Durante este proceso:

- Se diseñaran variables con capacidad explicativa (por ejemplo, ubicación, amenidades, comportamiento del host).
- Se combinan y transformarán variables existentes para capturar relaciones no evidentes.
- Se validará cada feature de manera individual para evaluar su distribución, coherencia y posible relación con la variable objetivo.

El objetivo no es aumentar el número de variables, sino capturar mejor la señal relevante: ubicación, características del listing, calidad percibida y contexto del entorno. Estas features se desarrollaran de forma modular para facilitar su validación, reutilización y posterior integración en el pipeline de modelado.

Es importante destacar que en esta etapa no se realizan transformaciones dependientes del dataset (como normalización o imputación). Estas se manejarán posteriormente en la etapa de transformación de características, para asegurar consistencia entre entrenamiento e inferencia. El resultado de esta sección será conjunto de variables (X) derivadas más expresivas, listas para ser transformadas y utilizadas por el modelo.

Las funciones utilizadas en esta sección fueron previamente modularizadas dentro de la carpeta `src/features/build` del proyecto, lo que permite mantener una organización clara y reutilizable. El módulo `orchestrator` actuará como orquestador de la creación de features, integrando cada módulo en un flujo único y consistente. Este enfoque no solo mejora la calidad de las features, sino que también asegura que el proceso pueda escalar y aplicarse de forma consistente en entornos de entrenamiento e inferencia.

In [43]:
# Import validate features module
from src.features.analysis.diagnostics.validate_features import validate_features

# Import orchestrator to build feature set
from src.features.build.orchestrator import build_features

# Apply orchestrator → prepare dataset with constructed features
df_prepared = build_features(df_prepared)

# Display the first rows of the prepared dataset
df_prepared.head()

,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,beds,minimum_nights,...,has_carbon_monoxide_alarm,has_fire_extinguisher,has_exterior_security_cameras_on_property,has_first_aid_kit,amenity_score,host_verifications_grouped,host_total_listings_segment,review_scores_mean,has_review,review_scores_mean_segment
0,Cuajimalpa de Morelos,19.38283,-99.27178,Entire villa,entire_home/apt,2,1.0,1.0,1.0,1,...,0.0,0.0,0.0,0.0,0.268380,extended,small_host,NaN,0.0,NaN
1,Cuauhtémoc,19.41162,-99.17794,Entire home,entire_home/apt,14,5.5,5.0,8.0,1,...,1.0,0.0,1.0,0.0,1.044497,extended,large_host,4.707143,1.0,low_review
2,Cuauhtémoc,19.43977,-99.15605,Entire condo,entire_home/apt,2,1.0,1.0,1.0,15,...,0.0,0.0,0.0,0.0,0.976673,basic,medium_host,4.881429,1.0,medium_review
3,Miguel Hidalgo,19.40826,-99.18659,Entire home,entire_home/apt,16,5.0,5.0,10.0,2,...,0.0,0.0,0.0,1.0,1.261502,basic,medium_host,4.868571,1.0,medium_review
4,Benito Juárez,19.39675,-99.17581,Private room in rental unit,private_room,2,1.0,1.0,1.0,4,...,0.0,0.0,1.0,1.0,1.033924,basic,medium_host,4.862857,1.0,medium_review


Una vez creadas las variables derivadas, verificaremos que hayan sido correctamente construidas y que mantengan coherencia con los hallazgos obtenidos durante el análisis exploratorio. Este proceso de validación se aplicará de manera sistemática a todas las features derivadas.

El propósito de estas validaciones no es reinterpretar los resultados del EDA, sino garantizar:

- Coherencia en la distribución de las variables respecto a lo observado previamente.
- Ausencia de valores nulos inesperados que puedan comprometer la calidad del dataset.
- Consistencia estadística en métricas clave (media, varianza, skewness, etc.).
- Presencia de señal preliminar respecto a la variable objetivo, confirmando que las features aportan valor predictivo.

De esta forma, las validaciones aseguran que las variables derivadas sean confiables, reproducibles y listas para integrarse en el pipeline de modelado sin introducir sesgos ni inconsistencias.

### Features de ubicación (Geo)

La ubicación es uno de los factores más determinantes en el precio de un listing. Sin embargo, variables como latitud y longitud por sí solas no capturan de forma directa el valor contextual de una zona. y no son directamente interpretables por el modelo. 

Se construyeron features geoespaciales que capturan accesibilidad y atractivo del entorno, como la cercanía a zonas turísticas o culturales y la densidad de actividad comercial en la zona. Con estas variables buscaremos traducir la ubicación en señales más informativas y comparables entre listings.

Entre las variables derivadas se incluyeron:

- `distance_to_attractions`: mide la proximidad mínima de un listing a la atracción cultural o turística más cercana.

- `attractions_density`: cuenta el número de atracciones dentro de un radio definido alrededor del listing, reflejando la concentración cultural o turística.

- `commercial_density`: cuenta el número de puntos de interés comerciales (cafés, restaurantes, bares, etc.) en el mismo radio, como proxy de actividad económica y conveniencia.

Estas variables permitirán traducir coordenadas geográficas en señales más interpretables y útiles para el modelo. Para garantizar reutilización y claridad en el pipeline, la lógica geoespacial fue encapsulada en el módulo `features/build/geo.py`, donde se implementaron funciones eficientes basadas en estructuras como BallTree para el cálculo de distancias y conteos espaciales. 



In [44]:
# Display the first rows of latitude, longitude, and the new geo features
geo_features = ['dist_to_nearest_attraction', 'attractions_within_radius', 'commercial_within_radius']
df_prepared[["latitude", "longitude"] + geo_features]

,latitude,longitude,dist_to_nearest_attraction,attractions_within_radius,commercial_within_radius
0,19.382830,-99.271780,2.450990,0,1
1,19.411620,-99.177940,0.415358,4,252
2,19.439770,-99.156050,0.375601,3,129
3,19.408260,-99.186590,0.985700,1,53
4,19.396750,-99.175810,1.557305,0,116
...,...,...,...,...,...
21445,19.442240,-99.113440,2.178589,0,14
21446,19.308017,-99.168158,0.958782,1,29
21447,19.434460,-99.174010,0.832345,1,128
21448,19.406435,-99.160934,0.958943,1,184


In [45]:
# Validate geo features
validate_features(df_prepared, geo_features, 'log_price')


========== Validating: dist_to_nearest_attraction ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0   float64  21450     21450   18042     0           0.0

--- Statistical Summary ---
                          0
mean               1.303117
median             0.715394
mode               0.606966
std                1.591586
min                0.004748
p25                0.394563
p50                0.715394
p75                1.571273
max               16.244343
skew               2.970778
kurt              11.932050
outliers_count  1840.000000
outliers_pct       8.580000
pearson_r         -0.239464

========== Validating: attractions_within_radius ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0     int64  21450     21450      12     0           0.0

--- Statistical Summary ---
                        0
mean             2.719347
median           2.000000
mode             0.000000
std              2.79


**1. `dist_to_nearest_attraction`**

- La distribución presenta una fuerte asimetría positiva, lo cual es consistente con la naturaleza geográfica del problema (muchas propiedades cercanas a puntos de interés y pocas muy alejadas).
- No se observan valores nulos, lo que confirma que el cálculo de distancias fue aplicado correctamente a todo el dataset.
- La correlación negativa con el precio (pearson_r ≈ -0.24) es coherente con lo esperado: a mayor distancia, menor valor.
- Se identifican outliers, pero estos representan casos extremos válidos (propiedades alejadas), no errores de cálculo.
- La feature está correctamente construida, con señal consistente y sin anomalías estructurales.

**2. `attractions_within_radius`**

- Variable discreta con baja cardinalidad, lo cual facilita su interpretabilidad.
- La distribución muestra concentración en valores bajos, indicando que muchas propiedades tienen pocos puntos de interés cercanos.
- La correlación positiva con el precio (pearson_r ≈ 0.30) confirma la hipótesis inicial: mayor densidad de atracciones → mayor valor.
- No presenta outliers ni valores inconsistentes.
- La feature es estable, interpretable y alineada con los patrones observados en EDA.

**3. `commercial_within_radius`**

- Presenta una distribución más amplia, reflejando mayor variabilidad en zonas comerciales.
- La correlación positiva (pearson_r ≈ 0.29) indica que la actividad comercial cercana es un factor relevante en el precio.
- La ausencia de outliers sugiere que la variable captura bien la densidad sin distorsiones extremas.
- Alta cardinalidad, lo que podría requerir transformación posterior dependiendo del modelo.
- Feature con buena señal y variabilidad, candidata fuerte para el modelo, sujeta a posible transformación.

En general, las features de ubicación muestran consistencia con el análisis exploratorio previo y capturan correctamente el contexto espacial de las propiedades, aportando información relevante para el modelado.

### Features de tipo de propiedad

Las variable `property_type` presenta una alta granularidad (por ejemplo, múltiples variantes de apartamentos, casas, habitaciones, etc.), lo que dificulta su uso directo en el modelo y puede introducir ruido.

Se construyeron las variables derivadas de `property_type` y `room_type` que simplifican y estructuran esta información, manteniendo su significado esencial:

- `property_group`: agrupa los distintos tipos de propiedad en categorías más generales (apartamento, casa, hotel, etc.), reduciendo la cardinalidad y mejorando la interpretabilidad.
- `property_group_room`: combina el grupo de la propiedad con el tipo de habitación, capturando interacciones relevantes entre ambos factores.

Esta segunda variable resulta especialmente útil, ya que permite diferenciar escenarios que, aunque comparten el mismo tipo de propiedad, tienen comportamientos distintos en función del tipo de ocupación. El objetivo de estas creaciones fue transformar variables categóricas complejas en representaciones más compactas y expresivas, facilitando su posterior codificación y uso en el modelo.

La lógica de estas transformaciones fue modularizada en el modulo `features/build/property.py`, permitiendo su reutilización dentro del pipeline de `build_features`.




In [46]:
# Display the first rows of property features
property_features = ['room_type','property_group', 'property_group_room']
df_prepared[property_features]

,room_type,property_group,property_group_room
0,entire_home/apt,house,house_entire_home/apt
1,entire_home/apt,house,house_entire_home/apt
2,entire_home/apt,apartment,apartment_entire_home/apt
3,entire_home/apt,house,house_entire_home/apt
4,private_room,apartment,apartment_private_room
...,...,...,...
21445,private_room,apartment,apartment_private_room
21446,private_room,apartment,apartment_private_room
21447,entire_home/apt,apartment,apartment_entire_home/apt
21448,entire_home/apt,apartment,apartment_entire_home/apt


In [47]:
# Validate property features
validate_features(df_prepared, property_features, 'log_price')


========== Validating: room_type ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0       str  21450     21450       4     0           0.0

--- Frequency distribution ---
                 count    pct
room_type                    
entire_home/apt  14749  68.76
private_room      6423  29.94
shared_room        231   1.08
hotel_room          47   0.22

--- Median target by category ---
         room_type  median_target
0       hotel_room       7.600902
1  entire_home/apt       7.133296
2     private_room       6.363028
3      shared_room       5.579730

========== Validating: property_group ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0       str  21450     21450       6     0           0.0

--- Frequency distribution ---
                count    pct
property_group              
apartment       16566  77.23
house            2880  13.43
hotel             987   4.60
guesthouse        803   3.74
unique/n

**1. `room_type`**

- Variable categórica con baja cardinalidad (4 categorías) y sin valores nulos.
- La distribución está claramente desbalanceada, con predominancia de propiedades completas (~69%), seguida de habitaciones privadas (~30%).
- La variable captura bien diferencias de precio entre modalidades de alojamiento, con un gradiente claro de hotel > entire home > private room > shared room.
- Feature estable, con buena interpretabilidad y señal clara, alineada con la lógica del negocio.

**2. `property_group`**

- La agrupación reduce efectivamente la cardinalidad del tipo de propiedad original a 6 categorías.
- La distribución está dominada por apartamentos (~77%), lo cual refleja la composición del dataset.
- Se mantiene una diferenciación en términos de precio entre categorías, aunque con menor separación que en `room_type`.
- Feature correctamente agregada, útil para reducir complejidad sin perder completamente la señal original.

**3. `property_group_room`**

- Variable resultante de la interacción entre `property_group` y `room_type`, con mayor granularidad (19 categorías).
- Permite capturar diferencias relevantes dentro de una misma categoría de propiedad (por ejemplo, propiedad completa vs. habitación privada dentro de apartamentos).
- Se observa una mayor dispersión en los valores del target, lo que sugiere que esta variable captura interacciones más específicas.
- Algunas categorías presentan baja frecuencia, lo cual podría requerir tratamiento dependiendo del modelo.
- Feature más expresiva que las anteriores, con potencial para capturar relaciones más complejas, sujeta a evaluación posterior en la etapa de selección de variables.

En conjunto, las variables de tipo de propiedad muestran consistencia estructural, ausencia de anomalías y una relación coherente con la variable objetivo, quedando listas para su posterior transformación y evaluación dentro del modelo.

### Features de restricciones de reserva

Las variables relacionadas con las restricciones de reserva pueden reflejar el tipo de estancia que un anfitrión busca atraer, lo cual puede influir indirectamente en el precio de la propiedad. Durante el análisis exploratorio se evaluaron tanto `minimum_nights` como `maximum_nights`. Sin embargo, se observó que `maximum_nights` no aportaba información relevante para explicar la variación del precio, por lo que se decidió no incluirla en el proceso de feature engineering.

En cambio, `minimum_nights` mostró patrones más claros, por lo que se utilizó para construir una nueva variable categórica:

- `minimum_nights_segment`: agrupa las propiedades en distintos tipos de estancia (corta, media y larga duración), utilizando rangos definidos que representan comportamientos típicos de reserva.

Esta transformación permitió capturar de forma más interpretable la intención del anfitrión en términos de duración de estancia, simplificando una variable numérica en categorías con significado práctico y facilitando su uso posterior en el modelo.

La lógica de esta transformación fue encapsulada en el módulo `features/build/booking_restrictions.py`, permitiendo su reutilización dentro del pipeline de `build_features`.

In [48]:
# Display the first rows of booking restrictions features
booking_restrictions_features = ['minimum_nights', 'minimum_nights_segment']
df_prepared[booking_restrictions_features]

,minimum_nights,minimum_nights_segment
0,1,short_stay
1,1,short_stay
2,15,medium_stay
3,2,short_stay
4,4,medium_stay
...,...,...
21445,1,short_stay
21446,1,short_stay
21447,1,short_stay
21448,1,short_stay


In [49]:
# Validate booking restrictions features
validate_features(df_prepared, booking_restrictions_features, 'log_price')


========== Validating: minimum_nights ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0     int64  21450     21450      52     0           0.0

--- Statistical Summary ---
                          0
mean               3.171702
median             1.000000
mode               1.000000
std               13.409401
min                1.000000
p25                1.000000
p50                1.000000
p75                2.000000
max              365.000000
skew              20.905500
kurt             521.627656
outliers_count  2175.000000
outliers_pct      10.140000
pearson_r          0.029710

========== Validating: minimum_nights_segment ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0    object  21450     21450       3     0           0.0

--- Frequency distribution ---
                        count    pct
minimum_nights_segment              
short_stay              19275  89.86
medium_stay              1

**1. `minimum_nights`**

- Variable numérica con alta asimetría positiva, lo cual es consistente con su naturaleza: la mayoría de las propiedades permiten estancias cortas, mientras que existen pocos casos con valores muy altos.
- Se observan valores extremos (hasta 365 noches), lo que genera una alta curtosis y presencia de outliers (~10%). Estos valores, aunque extremos, representan casos válidos dentro del contexto del problema.
- La correlación con el precio es prácticamente nula (pearson_r ≈ 0.03), lo que indica que la variable en su forma original no captura una relación clara con la variable objetivo.
- Feature variable con distribución altamente sesgada y baja señal directa, lo que justifica su transformación.

**2. `minimum_nights_segment`**

- Variable categórica con baja cardinalidad (3 categorías) y sin valores nulos.
- La distribución está fuertemente concentrada en estancias cortas (~90%), reflejando el comportamiento predominante del mercado.
- Se observa una diferenciación en términos de precio entre segmentos, lo que sugiere que la agrupación captura patrones que no eran evidentes en la variable original.
- La transformación reduce la influencia de valores extremos y facilita la interpretación.

### Features de amenidades y servicios

Las amenidades y servicios disponibles en una propiedad representan uno de los componentes mas relevantes en la percepción de valor por parte de los usuarios, ya que reflejan el nivel de confort, funcionalidad y experiencia ofrecida.

Durante el análisis exploratorio se identificó que la información contenida en la lista de amenidades tiene un alto potencial predictivo, aunque se encuentra en un formato no estructurado. Por ello, en esta sección se construyeron diversas variables derivadas que permitieron transformar esta información en representaciones más útiles para el modelo.

Este enfoque permite capturar tanto información global como específica sobre las amenidades, combinando simplicidad e interpretabilidad con mayor capacidad de representación.

Las variables derivadas creadas incluyen:

- `amenities_count`:  medida general del nivel de equipamiento.
- `has_amenity`: columnas bonarias que indican la presencia de amenidades específicas relevantes.
- `amenities_score`: resumen el impacto combinado de múltiples amenidades en una sola métrica.

La lógica de estas transformaciones fue encapsulada en el módulo `features/build/amenities.py`, asegurando su reutilización dentro del pipeline de `build_features`.

#### Creación de `amenity_count`

Como primer paso en la creación de variables relacionadas con amenidades, se construyó una métrica simple pero informativa:

- `amenities_count`: representa el número total de amenidades disponibles en cada propiedad, calculado a partir de la lista original.

Esta variable permite capturar el nivel general de equipamiento de forma directa, transformando información no estructurada en una señal cuantitativa interpretable. Sin embargo, dado que la relación entre el número de amenidades y el precio presenta una relación lineal moderada, se contempló y analizó una versión discretizada de esta variable en el análisis exploratorio, la cual será generada en la etapa de transformación de características.

Con la creación de `amenity_count` establecemos una base cuantitativa sobre la cual se construirán representaciones más específicas del impacto de las amenidades.

In [50]:
# Display the first rows of amenity_count
df_prepared[['amenities', 'amenity_count']]

,amenities,amenity_count
0,"[Garden view, Resort access, Washer, Courtyard...",12
1,"[Piano, Patio or balcony, Wifi, Refrigerator, ...",26
2,"[Self check-in, Dining table, Elevator, Wifi, ...",28
3,"[Pack ’n play/Travel crib, Self check-in, Heat...",47
4,"[Elevator, Wifi, Hot water, Microwave, Refrige...",24
...,...,...
21445,"[Private entrance, Laundromat nearby, Dining t...",28
21446,"[Washer, Iron, Kitchen, Clothing storage, Hang...",9
21447,"[Paid parking off premises, Private entrance, ...",17
21448,"[Exterior security cameras on property, Kitche...",11


**1. Perfil general de `amenity_count`**

- Variable numérica sin valores nulos, lo que confirma que el proceso de parsing y conteo fue aplicado correctamente a todo el dataset.
- En algunos listings se obtuvieron listas vacías [], las cuales se traducen en un conteo igual a 0, indicando propiedades sin amenidades declaradas.
- Presenta una distribución relativamente equilibrada, con baja asimetría (skew ≈ 0.07) y curtosis cercana a una distribución normal.
- La dispersión es moderada, reflejando una variabilidad razonable en el nivel de equipamiento entre propiedades.
- Se identifican pocos valores extremos (~0.3%), lo que sugiere una distribución estable y sin anomalías significativas.

**2. Relación con la variable objetivo**

- La correlación positiva con el precio (pearson_r ≈ 0.34) indica una relación consistente entre mayor nivel de equipamiento y mayor valor de la propiedad.
- Esta señal resulta especialmente relevante considerando que se trata de una métrica agregada simple.

La variable `amenities_count` presenta una estructura estadística sólida, buena cobertura y una señal preliminar clara respecto a la variable objetivo: la cantidad total de amenidades si captura valor real. Su comportamiento sugiere que constituye una base robusta para representar el nivel general de equipamiento, además de servir como insumo para transformaciones posteriores como su discretización en bins.

#### Creación de columnas binarias `has_amenity`

Si bien el conteo total de amenidades permite capturar el nivel general de equipamiento, no todas las amenidades aportan el mismo valor. Algunas tienen un impacto más significativo en la percepción del usuario y, potencialmente, en el precio.

Por esta razón, en esta sección se construyeron variables binarias del tipo:

- `has_amenity`: indican la presencia o ausencia de amenidades específicas dentro de cada propiedad.

Estas variables permiten modelar el efecto individual de servicios concretos (por ejemplo, aire acondicionado, estacionamiento, televisión, etc.), capturando información que se pierde al utilizar únicamente métricas agregadas.

Para su construcción, se realizó un proceso de limpieza y normalización de la lista de amenidades, seguido de la identificación de un conjunto de servicios relevantes. A partir de este conjunto, se generaron columnas binarias que reflejan la disponibilidad de cada amenidad.

In [51]:
# Collect all columns starting with 'has_'
has_amenity_cols = [col for col in df_prepared.columns if col.startswith("has_")]

# Display the first rows of has_amenity columns
df_prepared[has_amenity_cols]

,has_washer,has_pool,has_streaming_platform,has_wifi,has_kitchen,has_hot_water,has_essentials,has_bed_linens,has_microwave,has_refrigerator,...,has_sauna,has_city_skyline_view,has_outdoor_furniture,has_outdoor_dining_area,has_smoke_alarm,has_carbon_monoxide_alarm,has_fire_extinguisher,has_exterior_security_cameras_on_property,has_first_aid_kit,has_review
0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0
2,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0
4,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21445,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
21446,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
21447,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
21448,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0


In [52]:
# Validate has_amenity features

# 1. Initialize a list to store results
results = []

# 2. Compute amenity-level statistics:
# For each binary amenity feature, calculate prevalence, median price differences,
# relative impact on price, and correlation with log_price.
for col in has_amenity_cols:
    # Proportion of listings that have the amenity
    proportion = df_prepared[col].mean()
    
    # Median log_price for listings with and without the amenity
    median_true = df_prepared.loc[df_prepared[col] == 1.0, "log_price"].median()
    median_false = df_prepared.loc[df_prepared[col] == 0.0, "log_price"].median()
    
    # Difference in medians (impact of having the amenity, en log)
    median_diff = median_true - median_false
    
    # Relative differences in price (%)
    real_median_diff = np.exp(median_diff) - 1

    # Correlation between amenity (binary) and log_price
    corr = df_prepared[col].corr(df_prepared["log_price"])
    
    # Append results as a dictionary
    results.append({
        "feature": col,
        "proportion": proportion,
        "median_true": median_true,
        "median_false": median_false,
        "median_diff": median_diff,
        "real_median_diff": f"{real_median_diff*100:.3f}%",
        "correlation_r": corr
    })

# 3. Convert results into a DataFrame
impact_has_amenity = pd.DataFrame(results).set_index("feature")

# 4. Display the final table
impact_has_amenity.sort_values(by="proportion", ascending=False).round(3)

,proportion,median_true,median_false,median_diff,real_median_diff,correlation_r
feature,,,,,,
has_wifi,0.990,6.957,6.867,0.090,9.375%,-0.001
has_kitchen,0.892,6.985,6.667,0.318,37.405%,0.086
has_review,0.884,6.946,7.090,-0.144,-13.417%,-0.071
has_tv,0.860,7.026,6.320,0.706,102.520%,0.286
has_hot_water,0.844,6.987,6.712,0.276,31.752%,0.096
has_cooking_basics,0.751,7.014,6.709,0.305,35.693%,0.138
has_dishes_and_silverware,0.743,7.006,6.745,0.261,29.765%,0.125
has_iron,0.742,7.032,6.666,0.366,44.204%,0.192
has_essentials,0.717,7.015,6.763,0.252,28.671%,0.140


Dado el número de variables generadas, se utiliza un enfoque agregado para evaluar su calidad, considerando métricas como proporción, diferencias en la variable objetivo y correlación.

**1. Perfil general**

- Las variables presentan proporciones diversas, desde amenidades casi universales (por ejemplo, `has_wifi`) hasta otras poco frecuentes (por ejemplo, `has_piano`, `has_sauna`).
- Esta variabilidad en frecuencia es esperada y refleja la heterogeneidad en el nivel de equipamiento entre propiedades.
- No se observan inconsistencias estructurales en la construcción de las variables.

**2. Relación con la variable objetivo**

- En general, múltiples amenidades muestran diferencias positivas en la mediana del precio cuando están presentes, lo que sugiere una contribución potencial al valor de la propiedad.
- Algunas variables presentan correlaciones moderadas (por ejemplo, `has_tv`, `has_hair_dryer`, `has_elevator`, `has_free_parking`), indicando señal consistente.
- También se identifican amenidades con bajo o nulo impacto aparente, así como algunos casos con diferencias negativas, lo cual es esperable dada la diversidad y frecuencia de estas variables.

**3. Consideraciones**

- Amenidades con proporciones muy altas (cercanas a 1) o muy bajas (cercanas a 0) tienden a aportar menor información efectiva al modelo.
- Algunas variables con alto impacto aparente pueden estar influenciadas por baja frecuencia, por lo que su efecto deberá evaluarse con mayor cuidado.
- A partir de estos criterios, se considera selccionar un subconjunto de variables que combine relevancia estadística, interpretabilidad y frecuencia suficiente, con el objetivo de maximizar la señal predictiva y reducir el riesgo de ruido, redundancia o sobreajuste.

Las variables binarias `has_amenity` fueron correctamente construidas y presentan un comportamiento coherente con lo observado durante el análisis exploratorio. Estas variables permiten capturar señales específicas que complementan las métricas agregadas, y serán evaluadas de forma conjunta en la etapa de selección de variables y construcción del modelo.

#### Creación de `amenity_score`

Las variables binarias (`has_amenity`) permiten capturar el efecto individual de cada servicio, pero su uso directo puede introducir alta dimensionalidad y fragmentación de la señal.

Para abordar esto, en esta sección se construyó una variable agregada:

- `amenity_score`: score numérico que resume el impacto conjunto de múltiples amenidades en una sola métrica.

El proceso de construcción del score sigue tres pasos principales:

- Selección de amenidades relevantes, basada en criterios de frecuencia y relación con la variable objetivo.
- Asignación de pesos, calculados como la combinación de la diferencia de medianas y la correlación (*median_diff * correlation*). Se multiplican ambas métricas para obtener un peso que refleja tanto el impacto como la consistencia estadística de la relación.
- Cálculo del score, mediante la suma de los pesos de las amenidades presentes en cada propiedad.

$$
\text{amenity\_score}_{listing} = \sum_{i=1}^{n} \big( \text{has\_amenity}_{i} \cdot w_{\text{amenidad i}} \big)
$$


De esta forma, `amenitiy_score` permite consolidar múltiples señales en una representación más compacta y expresiva. Reduciendo dimensionalidad sin perder información relevante, capturando el valor agregado de las amenidades de forma más eficiente.

In [53]:
# Display the first rows of has_amenities columns
df_prepared['amenity_score']

0        0.268380
1        1.044497
2        0.976673
3        1.261502
4        1.033924
           ...   
21445    0.649838
21446    0.409493
21447    0.693755
21448    0.286073
21449    0.286073
Name: amenity_score, Length: 21450, dtype: float64

In [54]:
# Validate amenity_score feature
validate_features(df_prepared, ['amenity_score'], 'log_price')


========== Validating: amenity_score ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0   float64  21450     21450   11684     0           0.0

--- Statistical Summary ---
                       0
mean            0.937311
median          1.001458
mode            0.227788
std             0.388938
min             0.000000
p25             0.662755
p50             1.001458
p75             1.234661
max             1.874167
skew           -0.400846
kurt           -0.628696
outliers_count  0.000000
outliers_pct    0.000000
pearson_r       0.429551


**1. Perfil estadístico**

- Variable numérica sin valores nulos, lo que confirma la correcta aplicación del proceso de agregación.
- Presenta una distribución relativamente equilibrada, con ligera asimetría negativa (skew ≈ -0.40), indicando una mayor concentración de propiedades con scores altos.
- La dispersión es moderada, permitiendo diferenciar adecuadamente entre distintos niveles de equipamiento.
- No se observan outliers, lo que sugiere que el proceso de construcción del score es estable y robusto.

**2. Relación con la variable objetivo**

- La correlación con el precio (*pearson_r ≈ 0.43*) es considerablemente más alta que la observada en variables individuales como `amenities_count` o las variables binarias.
- Esto indica que el score logra capturar de manera más efectiva el impacto conjunto de las amenidades en el valor de la propiedad.

La variable `amenities_score` presenta una estructura estadística sólida y una señal clara respecto a la variable objetivo. Su comportamiento sugiere que esta variable logra consolidar información dispersa en una representación más compacta y expresiva, convirtiéndose en una de las features más prometedoras dentro del conjunto de amenidades.

### Features del anfitrión

Las características del anfitrión pueden influir en la percepción de confianza y calidad por parte de los usuarios, lo que potencialmente puede impactar en el precio de la propiedad.

Se construyeron dos variables derivada a partir de la información disponible del anfitrión:

- `host_verifications_grouped`: agrupa los distintos tipos de verificación en categorías más interpretables (por ejemplo, sin verificación, básica, extendida).
- `host_total_listings_segment`: clasifica a los anfitriones según el número total de propiedades que administran.

Dado que la variable original (`host_verifications`) se encuentra en un formato no estructurado, se realizó un proceso de transformación para convertirla en una representación categórica más compacta. Esta agrupación permite capturar distintos niveles de verificación y confianza del anfitrión de forma más clara, lo que también facilita su uso posterior en el modelo.

La segmentación de `host_total_listings_count` aportará una visión adicional sobre el perfil del anfitrión. Los anfitriones con mayor número de listados tienden a comportarse de manera distinta en términos de precios y estrategias de mercado frente a los anfitriones con pocas o con una sola propiedad. Esta variable segmentada facilitará la comparación entre distintos tipos de anfitriones y permitirá capturar patrones de profesionalización dentro de la plataforma, lo que puede resultar relevante para explicar variaciones en la variable objetivo.

La lógica de esta transformación fue encapsulada en el módulo dedicado `features/build/host.py`, permitiendo su reutilización dentro del pipeline de `build_features`.

In [55]:
# Display the first rows of has_amenities columns
df_prepared[['host_verifications', 'host_verifications_grouped', 'host_total_listings_count', 'host_total_listings_segment']]

,host_verifications,host_verifications_grouped,host_total_listings_count,host_total_listings_segment
0,"[email, phone, work_email]",extended,1.0,small_host
1,"[email, phone, work_email]",extended,13.0,large_host
2,"[email, phone]",basic,5.0,medium_host
3,"[email, phone]",basic,6.0,medium_host
4,"[email, phone]",basic,3.0,medium_host
...,...,...,...,...
21445,"[email, phone]",basic,5.0,medium_host
21446,"[email, phone]",basic,1.0,small_host
21447,"[email, phone]",basic,24.0,professional_host
21448,[phone],low,7.0,large_host


In [56]:
# Validate host features
validate_features(df_prepared, ['host_verifications_grouped', 'host_total_listings_segment'], 'log_price')


========== Validating: host_verifications_grouped ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0       str  21450     21450       4     0           0.0

--- Frequency distribution ---
                            count    pct
host_verifications_grouped              
basic                       16682  77.77
extended                     2684  12.51
low                          2072   9.66
none                           12   0.06

--- Median target by category ---
  host_verifications_grouped  median_target
0                   extended       7.102499
1                      basic       6.937314
2                        low       6.889082
3                       none       6.735312

========== Validating: host_total_listings_segment ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0    object  21448     21448       4     2          0.01

--- Frequency distribution ---
                             count  

La variable `host_verifications_grouped` fue correctamente construida y presenta un comportamiento coherente con lo esperado. Aunque su impacto individual es limitado, aporta una señal contextual relacionada con la confianza y perfil del anfitrión, lo que podría complementar otras variables dentro del modelo.

**1. Perfil general**

- Variable categórica sin valores nulos y con baja cardinalidad (4 categorías), lo que facilita su interpretación y uso en modelos.
- La distribución está fuertemente concentrada en la categoría *basic* (~78%), seguida por *extended* y *low*, mientras que *none* representa una proporción marginal.
- Esta distribución es consistente con el comportamiento esperado en plataformas donde la mayoría de anfitriones cuentan con verificaciones básicas.

**2. Relación con la variable objetivo**

- Se observa una tendencia creciente en el precio conforme aumenta el nivel de verificación:
  - *extended* presenta la mediana más alta.
  - *basic* y *low* muestran valores intermedios.
  - "none" presenta la mediana más baja.
- Esto sugiere que un mayor nivel de verificación podría estar asociado con propiedades de mayor valor.

**3. Consideraciones**

- La categoría "none" tiene una frecuencia extremadamente baja, por lo que su comportamiento debe interpretarse con cautela.
- Las diferencias entre categorías son moderadas, lo que indica que la señal existe, pero no es dominante frente a otras variables del modelo.

La variable `host_total_listings_segment` aporta una señal distinta, enfocada en la escala de operación del anfitrión más que en su nivel de verificación.

**1. Perfil general**

- Variable categórica con baja cardinalidad (4 categorías) y prácticamente sin valores nulos (~0.01%), lo que facilita su interpretación y uso en modelos.
- La distribución está relativamente balanceada: small_host (27.9%), large_host (24.9%), professional_host (24.7%) y medium_host (22.6%).
- Esta segmentación refleja distintos perfiles de anfitriones, desde individuales con pocos listados hasta profesionales con múltiples propiedades.

**2. Relación con la variable objetivo**

- Se observa una tendencia creciente en la mediana del precio conforme aumenta el tamaño del portafolio del anfitrión:
  - *professional_host* presenta la mediana más alta (~7.16).
  - *large_host* y *small_host* muestran valores intermedios (~6.95 y ~6.87).
  - *medium_host* presenta la mediana más baja (~6.80).
- Esto sugiere que los anfitriones con mayor número de listados tienden a manejar propiedades de mayor valor o con estrategias de precio más agresivas.

**3. Consideraciones**

- Las diferencias entre categorías son moderadas, aunque la señal es más clara que en `host_verifications_grouped`, lo que indica que esta segmentación puede aportar mayor capacidad explicativa en el modelo.
- La variable captura un aspecto estructural del mercado: la profesionalización del anfitrión, que puede influir en la fijación de precios y en la calidad percibida de las propiedades.



### Features de puntuaciones de reseñas

Las puntuaciones de reseñas reflejan la experiencia de los usuarios en una propiedad y constituyen una señal directa de calidad percibida lo que podría incluir en el precio.

Se construyeron variables derivadas a partir de las distintas métricas de evaluación disponibles en el conjunto de datos:

- `review_scores_mean`: promedio de las diferentes dimensiones de evaluación (limpieza, comunicación, ubicación, etc.), con el objetivo de obtener una medida global de calidad.
- `has_review`: variable binaria que indica si la propiedad cuenta o no con reseñas.
- `review_scores_mean_segment`:  variable categórica que clasifica las propiedades en segmentos según el valor de su `review_scores_mean`

La variable `review_scores_mean` permite consolidar múltiples dimensiones en una representación única y más manejable, mientras que `has_review` captura la diferencia entre propiedades con historial de evaluaciones y aquellas sin información disponible. Finalmente, `review_scores_mean_segment` facilitará el análisis comparativo al agrupar las propiedades en categorías de calidad, lo que permitirá detectar patrones y diferencias más fácilmente.

Estas transformaciones permitirán incorporar tanto la calidad percibida como la disponibilidad de información, dos factores relevantes en la valoración de una propiedad.

La lógica de construcción fue modularizada en `features/build/reviews.py` para su integración dentro del pipeline de `build_features`.



In [57]:
# Display the first rows of review features
review_features = ['review_scores_mean', 'has_review', 'review_scores_mean_segment']
df_prepared[review_features]

,review_scores_mean,has_review,review_scores_mean_segment
0,NaN,0.0,NaN
1,4.707143,1.0,low_review
2,4.881429,1.0,medium_review
3,4.868571,1.0,medium_review
4,4.862857,1.0,medium_review
...,...,...,...
21445,NaN,0.0,NaN
21446,NaN,0.0,NaN
21447,NaN,0.0,NaN
21448,NaN,0.0,NaN


In [58]:
validate_features(df_prepared, review_features, 'log_price')


========== Validating: review_scores_mean ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0   float64  18964     18964    1168  2486         11.59

--- Statistical Summary ---
                          0
mean               4.798229
median             4.857143
mode               5.000000
std                0.274281
min                1.000000
p25                4.761429
p50                4.857143
p75                4.925714
max                5.000000
skew              -6.277359
kurt              60.217471
outliers_count  1176.000000
outliers_pct       6.200000
pearson_r          0.131331

========== Validating: has_review ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0   float64  21450     21450       2     0           0.0

--- Statistical Summary ---
                          0
mean               0.884103
median             1.000000
mode               1.000000
std                0.320109
min     

**1. `review_scores_mean`**

- La variable presenta valores faltantes (~11.6%), correspondientes a propiedades sin reseñas.
- La distribución está fuertemente sesgada hacia valores altos (~skew ≈ -6.29~), con una alta concentración cercana al máximo (5.0).
- La baja dispersión (std ≈ 0.27) indica que la mayoría de las propiedades tienen evaluaciones similares.
- Se observan outliers en la cola baja, correspondientes a propiedades con calificaciones significativamente menores.

Respecto a la relación con la variable objetivo:

- La correlación con el precio es baja (pearson_r ≈ 0.13), lo que sugiere una relación limitada.
- Esto puede explicarse por la falta de variabilidad en la variable, ya que la mayoría de las propiedades tienen puntuaciones altas.


**2. `has_review`**

- Variable binaria sin valores nulos.
- La mayoría de las propiedades cuentan con reseñas (~88%), mientras que una proporción menor (~12%) no tiene información disponible.
- Se observa que las propiedades sin reseñas presentan una mediana de precio ligeramente superior a aquellas con reseñas.
- Este comportamiento puede estar influenciado por factores externos, como propiedades nuevas.

**3. `review_scores_mean_segment`**

- Variable categórica con tres segmentos (low_review, medium_review, high_review) y un ~11.6% de valores nulos.
- La distribución es relativamente balanceada entre low_review (33.4%) y high_review (32.8%), con menor representación en medium_review (22.2%).
- La mediana del precio muestra una clara progresión: las propiedades con high_review tienen mayor precio (≈7.06), seguidas por medium_review (≈6.98) y finalmente low_review (≈6.81).
- Esto indica que la segmentación captura diferencias en la variable objetivo que no eran tan evidentes en el promedio continuo (review_scores_mean), aportando mayor capacidad discriminativa.

**3. Consideraciones**

- La variable `review_scores_mean` presenta saturación en valores altos, lo que limita su capacidad discriminativa.
- La señal observada en `has_review` no necesariamente implica causalidad directa, sino que puede reflejar efectos indirectos del mercado.
- La inclusión de `review_scores_mean_segment` permite superar parcialmente la limitación de la saturación, ya que introduce categorías que diferencian mejor el comportamiento del precio según la calidad percibida.

Las variables de reseñas fueron correctamente construidas y aportan información complementaria sobre la calidad percibida y la disponibilidad de evaluaciones. Sin embargo, su capacidad predictiva individual parece limitada, por lo que su utilidad deberá evaluarse en conjunto con el resto de variables durante la etapa de modelado.

### Punto de Control: Creación de `df_base_model`

Antes de iniciar la etapa de preprocesamiento, se generará una copia del dataset denominada `df_base_model`. Este punto de control tiene como objetivo preservar una versión de los datos que contenga todas las variables originales y derivadas generadas durante la fase de Feature Engineering, pero sin transformaciones dependientes de los datos.

La creación de este dataset responde a los siguientes motivos:

- El dataset `df_prepared` será utilizado para aplicar técnicas de preprocesamiento como imputación de valores faltantes, transformaciones numéricas, codificación de variables categóricas y escalamiento. Estas transformaciones se aplicarán con el propósito de facilitar el análisis, evaluación y selección final de variables.

- Algunas de estas técnicas aprenden parámetros a partir de los datos (por ejemplo, medias, medianas, frecuencias, categorías o escalas). Por este motivo, no deben formar parte del conjunto de datos que posteriormente será utilizado para entrenar los modelos finales.

- Dado que varias transformaciones modificarán o reemplazarán la representación original de las variables, el dataset resultante no será adecuado como punto de partida para la fase de modelado.

- Una vez completada la evaluación y selección de variables, el conjunto final de features será extraído directamente desde `df_base_model` para construir `df_model`. De esta forma, las variables seleccionadas conservarán su estado previo al preprocesamiento dependiente de los datos.

Este enfoque permitirá separar claramente la fase de análisis y selección de variables de la fase de modelado. Además, garantiza que el pipeline de preprocesamiento pueda ajustarse posteriormente utilizando únicamente los datos de entrenamiento, evitando data leakage y asegurando la reproducibilidad del flujo completo de entrenamiento e inferencia.


In [59]:
# Save a control point before applying transformations
df_base_model = df_prepared.copy()

## 5. Preprocesamiento de Características (Preprocess Features)

Una vez construidas las variables derivadas, el siguiente paso consiste en preprocesar las características para hacerlas más adecuadas al modelado predictivo. Esta etapa busca ajustar la representación de los datos con el fin de mejorar la estabilidad estadística, la comparabilidad entre observaciones y la capacidad predictiva de las variables dentro del modelo.

El preprocesamiento puede abarcar diferentes operaciones según la naturaleza de cada variable: ajustes de escala, transformaciones de distribuciones sesgadas, manejo de valores faltantes, tratamiento de valores extremos y cambios en la representación categórica o numérica. El objetivo es garantizar que las características estén en un formato consistente y que aporten información útil al modelo.

Para organizar este proceso de manera clara y reproducible, la sección se dividirá en tres partes principales:

- **Diagnóstico de variables numéricas**: análisis de distribución, outliers, correlación con el target y comparación de transformaciones.
- **Diagnóstico de variables categóricas**: evaluación de cardinalidad, rareza de categorías yseñal predictiva.
- **Implementación de técnicas de preprocesamiento**: aplicación de imputaciones, transformaciones y escalados definidos.

Cada grupo de variables será tratado de manera independiente, aplicando técnicas específicas de preprocesamiento que se ajusten a sus características y al comportamiento observado en los datos. 

### Diagnóstico de variables numéricas

En esta sección se abordará el diagnóstico y evaluación de las variables numéricas, con el propósito de identificar sus características estadísticas y definir las transformaciones y tratamientos más adecuados para cada una.

Para ello, se seguirá un enfoque estructurado que combina dos etapas principales: **diagnóstico** y **comparación de transformaciones**. A partir del diagnóstico estadístico de cada variable, se analizan aspectos como la presencia de valores nulos, la forma de la distribución, la existencia de valores extremos y su relación con la variable objetivo.

Con base en este análisis, se generan sugerencias de tratamiento que permiten establecer, para cada variable:

- El método de imputación de valores nulos  
- La necesidad (o no) de aplicar transformaciones como logaritmos o winsorización  
- La conveniencia de realizar agrupaciones en bins para capturar posibles relaciones no lineales  
- El tipo de escalado más adecuado para su uso en modelos  

Adicionalmente, se realizará una comparación entre distintas transformaciones para aquellas variables que lo requieren, con el fin de evaluar su impacto y seleccionar la representación más adecuada.

A partir de estos resultados se determinarán las estrategias de imputación, transformación y escalado, con el fin de optimizar la representación de las variables y mejorar su utilidad en el modelo predictivo. Este enfoque permitirá tomar decisiones informadas y consistentes, evitando aplicar transformaciones de forma arbitraria y asegurando que cada variable sea tratada según su comportamiento específico en los datos.

In [60]:
# Import diagnostic function
from src.features.analysis.diagnostics.diagnose_features import build_numeric_diagnostics

# Define numerical features to analyze (including target)
numeric_features = [
    'dist_to_nearest_attraction', 'attractions_within_radius', 'commercial_within_radius',
    'accommodates', 'bathrooms', 'bedrooms', 'beds','amenity_count', 'amenity_score',
    'calculated_host_listings_count_entire_homes', 'calculated_host_listings_count_private_rooms', 
    'host_total_listings_count','minimum_nights','review_scores_mean',
    # target
    'log_price'
]


# Display diagnostics for numeric features (skewness, outliers, correlation, etc.)
print("\n ======================== DIAGNOSTIC OF NUMERICAL FEATURES ========================\n")
build_numeric_diagnostics(df_prepared, numeric_features)



 ======================== DIAGNOSTIC OF NUMERICAL FEATURES ========================



,feature,nulls_%,skew,kurtosis,std,min,max,outliers_%,corr_with_target,signal_suggestion,null_treatment_suggestion,transform_suggestion,binning_suggestion,scaling_suggestion
0,dist_to_nearest_attraction,0.000000,2.970778,11.932050,1.591586,0.004748,16.244343,0.085781,-0.239464,useful_signal,no_impute,log,no_binning,robust
1,attractions_within_radius,0.000000,0.676436,-0.617120,2.799769,0.000000,11.000000,0.000000,0.304176,useful_signal,no_impute,no_transform,no_binning,minmax
2,commercial_within_radius,0.000000,0.661381,-0.569216,142.318307,0.000000,552.000000,0.000000,0.294229,useful_signal,no_impute,no_transform,no_binning,minmax
3,accommodates,0.000000,2.500608,9.161223,2.295643,1.000000,16.000000,0.048718,0.591168,useful_signal,no_impute,log,no_binning,minmax
4,bathrooms,0.000559,4.651919,46.465707,0.896338,0.000000,21.000000,0.025515,0.387097,useful_signal,median,log,binning_candidate (low_variance/discrete),minmax
5,bedrooms,0.004522,6.197373,119.629315,1.047670,0.000000,40.000000,0.031143,0.501791,useful_signal,median,log,no_binning,minmax
6,beds,0.000746,4.894336,52.908574,1.585855,0.000000,40.000000,0.096016,0.444168,useful_signal,median,log,no_binning,robust
7,amenity_count,0.000000,0.070003,-0.427014,15.537805,0.000000,102.000000,0.003170,0.335596,useful_signal,no_impute,no_transform,no_binning,standard
8,amenity_score,0.000000,-0.400846,-0.628696,0.388938,0.000000,1.874167,0.000000,0.429551,useful_signal,no_impute,no_transform,binning_candidate (low_variance/discrete),standard
9,calculated_host_listings_count_entire_homes,0.000000,4.125971,18.317500,33.670600,0.000000,221.000000,0.143450,0.198454,useful_signal,no_impute,log,binning_candidate (extreme_values),robust


El diagnóstico realizado permite identificar patrones clave en el comportamiento de las variables numéricas, así como definir estrategias de transformación coherentes para su uso en el modelado.

**1. Presencia de señal en la mayoría de variables**

La gran mayoría de las variables numéricas presentan una correlación relevante con variable objetivo (`log_price`)  clasificándose como *useful_signal**. Este hallazgo confirma que el conjunto de variables no está compuesto por atributos irrelevantes, sino por características con potencial explicativo y predictivo. La diversidad de fuentes de señal (capacidad, ubicación, amenidades, host) también sugiere que el modelo podrá capturar distintos aspectos del fenómeno de precios.

**Excepciones identificadas:**
- `minimum_nights`: presenta una correlación prácticamente nula con el target (≈ 0.03), lo que indica que su aporte directo es limitado.  
- `host_total_listings_count`, `review_scores_mean`: muestran correlaciones débiles (≈ 0.11 - 0.13), clasificándose como *weak_signal*.  

Estas variables, aunque no deben descartarse de inmediato, requieren un tratamiento más cuidadoso. Su valor puede emerger tras transformaciones (logarítmicas, binning) o al ser combinadas con otras señales. Sin embargo, también se podría reconsiderar su inclusión en el modelo si tras pruebas no aportan mejora significativa, evitando así ruido o sobreajuste.

**2. Alta asimetría y presencia de outliers**

El diagnóstico revela un patrón consistente en múltiples variables caracterizado por:

- Alta skewness (asimetría positiva): distribuciones sesgadas hacia la derecha, donde la mayoría de observaciones se concentran en valores bajos y existen colas largas hacia valores altos.
- Alta kurtosis: presencia de colas pesadas y picos pronunciados, lo que indica una mayor probabilidad de valores extremos.
- Outliers significativos: proporciones relevantes de observaciones que se alejan del rango central, afectando la estabilidad estadística y el comportamiento de los modelos.

Este comportamiento es típico de variables con distribuciones de cola larga, donde predominan valores pequeños pero aparecen casos aislados con magnitudes muy superiores. Dicho patrón puede distorsionar coeficientes en modelos lineales, afectar métricas de distancia en algoritmos basados en similitud y ralentizar la convergencia en métodos de optimización.

Las variables más afectadas son:

- Variables de capacidad:  
  - `accommodates`, `beds`, `bedrooms`, `bathrooms`  
  Presentan alta asimetría y curtosis extrema, reflejando que la mayoría de alojamientos tienen valores bajos, pero existen casos con capacidades muy elevadas que generan colas largas.

- Variables relacionadas con hosts:  
  - `calculated_host_listings_count_*`, `host_total_listings_count`  
  Exhiben distribuciones altamente sesgadas, con la mayoría de hosts gestionando pocos anuncios y unos pocos con cientos de propiedades, lo que introduce valores extremos.

- Restricciones de estancia:   
  - `minimum_nights`  
  Es el caso más extremo, con skew ≈ 20 y curtosis > 500. La mayoría de listados tienen estancias mínimas cortas, pero algunos presentan valores desproporcionadamente altos (ej. cientos de noches), generando una distribución altamente no lineal.

**Implicaciones y tratamiento sugerido:**
- Aplicar transformaciones logarítmicas para reducir la asimetría y estabilizar la varianza.  
- Considerar escalado robusto (basado en mediana y rango intercuartílico) para mitigar el impacto de outliers.  
- Evaluar binning en variables discretas o con valores extremos recurrentes, como `bathrooms` o `minimum_nights`.  
- Validar el efecto de estas transformaciones en la correlación con el target, asegurando que la señal útil se preserve o incluso se potencie.

La alta asimetría y presencia de outliers es un rasgo estructural de varias variables. Su tratamiento adecuado será esencial para garantizar estabilidad en el modelado y evitar que valores extremos dominen el aprendizaje.

**3. Transformaciones logarítmicas como solución clave**

Dado el comportamiento anterior identificado en múltiples variables hace que la transformación logarítmica (`log`) se convierta en una herramienta fundamental dentro del pipeline de preprocesamiento. La transformación logaritmica resulta adecuada para:

- Reducir la asimetría, suavizar distribuciones sesgadas y acercárlas a formas más simétricas. 
- Comprimir valores extremos, los outliers pierden influencia relativa disminuyendo su impacto en el modelo.
- Linealizar relaciones, facilitando que las variables se relacionen de manera más proporcional con el target (`log_price`)
- Estabilizar la varianza, mejora la convergencia de algoritmos y reduce el riesgo de sobreajuste.

**Consideraciones prácticas:**
- Para evitar problemas con valores cero, se podría utilizar `log1p(x)` en lugar de `log(x)`, asegurando estabilidad en todas las observaciones.  
- La transformación debe validarse comparando correlaciones y ajustes antes y después, confirmando que la señal con el target se mantiene o mejora.  
- En algunos casos, la combinación de log con escalado robusto ofrece mayor estabilidad frente a outliers residuales.  


**4. Tipología de variables**

Se identifican cuatro grupos principales:

- Variables lineales y estables.

    - Características: baja asimetría, curtosis moderada, sin outliers relevantes, correlación útil con el target.
    - Ejemplos: amenity_count, attractions_within_radius, commercial_within_radius
    - Tratamiento: no requieren transformación, escalado estándar o MinMax.

- Variables con alta asimetría y curtosis.

    - Características: distribuciones sesgadas, colas largas, presencia de outliers, correlación significativa con el target.
    - Ejemplos: Variables de capacidad (accommodates, bathrooms, bedrooms, beds), Variables del host, minimum_nights.
    - Tratamiento: transformación logarítmica, escalado robusto, posible binning en casos discretos o extremos.

- Variables con señal débil o baja correlación.

    - Características: correlación baja con el target, distribuciones problemáticas (alta asimetría, curtosis extrema).
    - Ejemplos: host_total_listings_count (corr ≈ 0.13), review_scores_mean (corr ≈ 0.13), minimum_nights (corr ≈ 0.03).
    - Tratamiento: imputación por mediana, log-transform, binning para capturar patrones no lineales, validación de impacto en el modelo.

- Variables discretas o de baja varianza.

    - Características: valores concentrados en pocos niveles, poca dispersión, riesgo de baja señal si no se transforman.
    - Ejemplos: amenity_score, bathrooms (cuando toma valores enteros bajos), review_scores_mean (escala limitada 1–5)
    - Tratamiento: binning por categorías, escalado estándar o robusto según distribución.


**5. Estrategía de escalado**

El escalado constituye una etapa crítica en el preprocesamiento de variables numéricas, especialmente en modelos lineales y algoritmos sensibles a la magnitud de las características (ej. regresión, SVM, redes neuronales, KNN). Su objetivo es garantizar estabilidad numérica, mejorar la convergencia de los optimizadores y evitar que variables con rangos muy distintos dominen el aprendizaje.

La sugerencia del método de escalado en el diagnóstico se ha realizado de forma coherente con la naturaleza de cada variable:

- *MinMaxScaler* se asignó a variables acotadas o de conteo estable (`attractions_within_radius`, `commercial_within_radius`, `accommodates`, `bathrooms`, `bedrooms`), donde el rango es definido y los outliers no distorsionan significativamente. Este enfoque preserva la proporcionalidad y facilita la interpretación en modelos sensibles a rangos.

- *StandardScaler* se aplicó a variables con distribución aproximadamente normal y sin outliers relevantes (`amenity_count`, `amenity_score`). Al centrar en media cero y escalar a desviación estándar uno, se facilita la comparación entre variables y se estabilizan coeficientes en modelos lineales.

- *RobustScaler* se reservó para variables con alta asimetría, curtosis extrema y presencia de outliers (`dist_to_nearest_attraction`, `beds`, `host_total_listings_count`, `review_scores_mean`, `minimum_nights`). Este método utiliza la mediana y el rango intercuartílico, reduciendo la influencia de valores extremos y proporcionando mayor estabilidad.

**6. Consideraciones adicionales**

- `review_scores_mean`:   
  Aunque en el diagnóstico se sugiere la transformación logarítmica, esta variable está acotada en un rango limitado (1–5). Aplicar log no aporta beneficios significativos y puede distorsionar la escala. Es más adecuado tratarla como variable discreta o aplicar binning para capturar diferencias cualitativas en los niveles de puntuación.  En la sección de `build_features` se creó la variable segmentada `review_scores_mean_segment` que clasifica las propiedades en categorías de calidad y aporta mayor capacidad discriminativa frente al promedio continuo.

- `minimum_nights`:
  Presenta una distribución extremadamente sesgada y una correlación casi nula con el target. Más que una variable continua, su comportamiento es categórico: estancias cortas, medias y largas. Por ello, en la seccion de `build_features`se creó una variable segmentada mediante binning no lineal (ej. [1–7], [8–30], [31+]) para reflejar mejor su impacto en el modelo.

- `host_total_listings_count`:  
  Exhibe señal débil y valores extremos que generan alta asimetría. Puede ser útil tras una transformación logarítmica combinada con binning, pero también debe evaluarse si su inclusión aporta valor real al modelo. En caso de no mejorar el desempeño, podría considerarse su reducción o exclusión para evitar ruido. Para esta variable se construyó la variable segmentada `host_total_listings_segment`, que diferencia entre anfitriones según su escala de operación.

El diagnóstico no solo describe el comportamiento de las variables, sino que permite definir decisiones concretas de transformación. Esto marca la transición de un análisis exploratorio a un proceso de ingeniería de características orientado al modelado. Cada variable será transformada en función de su comportamiento específico, evitando transformaciones arbitrarias y asegurando una preparación adecuada de los datos para etapas posteriores.

### Comparación de transformaciones en variables numéricas

Con el objetivo de validar las sugerencias obtenidas en el diagnóstico, se realizará una comparación entre distintas transformaciones aplicadas a las variables numéricas seleccionadas. Esta comparación permite observar cómo cambian propiedades clave como la distribución (skewness), la dispersión (std) y la relación con la variable objetivo (corr_with_target).

In [61]:
# Import diagnostic function
from src.features.analysis.diagnostics.compare_features import compare_transformations

# Display comparison tables for different transformations applied to each feature
print("\n======================== COMPARISON BETWEEN TRANSFORMATIONS FOR EACH FEATURE ========================\n")
compare_transformations(df_prepared, numeric_features)


======================== COMPARISON BETWEEN TRANSFORMATIONS FOR EACH FEATURE ========================


========== VARIABLE: dist_to_nearest_attraction ==========

    transformation    mean     std  outliers_%    skew  corr_with_target
0         original  1.3031  1.5916      0.0858  2.9708           -0.2395
1              log  0.6883  0.4905      0.0337  1.1995           -0.2896
2             sqrt  1.0009  0.5489      0.0470  1.4085           -0.2811
3  winsor_moderate  1.2831  1.4823      0.0858  2.3745           -0.2505
4    winsor_strong  1.2043  1.2108      0.0858  1.6343           -0.2728

========== VARIABLE: attractions_within_radius ==========

    transformation    mean     std  outliers_%    skew  corr_with_target
0         original  2.7193  2.7998         0.0  0.6764            0.3042
1              log  0.9797  0.8540         0.0 -0.0084            0.3354
2             sqrt  1.2468  1.0793         0.0 -0.0025            0.3309
3  winsor_moderate  2.7187  2.7981         0.

Para cada variable se evaluaron las siguientes versiones:

- *Original* 
- *Log (`log1p`)*  
- *Raíz cuadrada (`sqrt`)*  
- *Winsorización moderada (1%)*  
- *Winsorización fuerte (5%)*  

El análisis se centró en tres aspectos principales:

- Reducción de asimetría (skewness) 
- Control de valores extremos (outliers) 
- mpacto en la correlación con el target  


**1. Transformaciones logarítmicas: las más efectivas en variables sesgadas.**

En variables con alta asimetría positiva (cola larga), se observa que la transformación logarítmica reduce significativamente el skew, comprime valores extremos y mantiene o mejora la correlación con el target. Esto se observa especialmente en las variables:
- `accommodates`
- `bathrooms`
- `beds`
- `bedrooms`
- `dist_to_nearest_attraction`
- `attractions_within_radius`
- variables relacionadas con el host

`log` es la transformación más consistente para este tipo de variables.

**2. Winsorización.**

La winsorización reduce el impacto de outliers sin alterar la forma general de la distribución y mantiene una estructura más cercana a la variable original. Sin embargo, se observa que no corrige la asimetría de fondo y su impacto en la correlación suele ser menor que el de `log`

Es útil cuando se quiere preservar interpretabilidad pero menos efectiva que `log` en distribuciones altamente sesgadas.

**3. Raíz cuadrada (`sqrt`).**

Para esta transformación se observa un solución intermedia. La transformación raíz reduce parcialmente la asimetría y tiene un efecto más suave que `log`. Pero suele quedarse corta en distribuciones con skew elevado y no siempre mejora la correlación.

`sqrt` es una alternativa cuando `log` es demasiado agresivo, pero para este conjunto de datos no parece ser la mejor opción.

**4. Variables estables.**

Para variables con baja asimetría, baja presencia de outliers y buena correlación, las transformaciones no aportan mejoras significativas. Esto se observa en variables como:
- `amenity_score`
- `amenity_count`
- `commercial_within_radius`

Se considerará mantener estas variables en su forma original.

**5. Transformación y escalado.**

A partir de la comparación realizada, es importante establecer el rol que cumple cada etapa dentro del tratamiento de variables numéricas. Las transformaciones (como `log`, `sqrt` o winsorización) se aplicarán con el objetivo de modificar la forma de la distribución, principalmente para reducir asimetría, controlar valores extremos y aproximar relaciones más lineales con la variable objetivo.

Sin embargo, estas transformaciones no resuelven las diferencias de escala entre variables. Por ello, una vez definida la mejor representación para cada feature, será necesario aplicar un escalado posterior.

El escalado de variables permite:

- Homogeneizar magnitudes entre variables  
- Evitar que aquellas con rangos más amplios dominen el modelo  
- Mejorar la estabilidad y convergencia en modelos sensibles a la escala (como regresión lineal)

En consecuencia, el flujo correcto en el pipeline de preprocesamiento consistirá en:

1. Seleccionar la mejor transformación para cada variable.
2. Aplicar dicha transformación.
3. Escalar todas las variables numéricas de forma consistent. 

Este enfoque asegurará que cada variable no solo tenga una distribución adecuada, sino también una escala compatible con el proceso de modelado.

### Diagnóstico de variables categóricas

Las variables categóricas representan atributos cualitativos del alojamiento que pueden contener información relevante para explicar el comportamiento del precio. Sin embargo, este tipo de variables no puede ser utilizado directamente por la mayoría de modelos de machine learning, especialmente modelos lineales, por lo que requieren un proceso de evaluación y transformación previo.

En esta sección se analizarán las variables categóricas (sin incluir las variables dicotómicas) candidatas al modelo con el objetivo de:

- Evaluar su calidad y estructura
- Analizar su cardinalidad y distribución
- Detectar categorías dominantes o poco frecuentes
- Evaluar su señal respecto al target
- Definir el tratamiento de valores faltantes
- Determinar la estrategia de codificación más adecuada para cada variable

Dependiendo de las características de cada feature, se considerarán diferentes estrategias de encoding, incluyendo:

- One-Hot Encoding
- Frequency Encoding
- Grouping of rare categories

El objetivo final es transformar las variables categóricas en representaciones numéricas que preserven la mayor cantidad posible de información útil para el modelo, manteniendo al mismo tiempo un equilibrio entre capacidad predictiva, estabilidad estadística e interpretabilidad.

In [62]:
# Import diagnostic function
from src.features.analysis.diagnostics.diagnose_features import build_categorical_diagnostics

# Define categorical features to analyze (including target)
categorical_features = [
    'neighbourhood_cleansed', 'room_type', 'property_group',
    'property_group_room','minimum_nights_segment', 'host_verifications_grouped',
    'review_scores_mean_segment', 'host_total_listings_segment',
    # target
    'log_price'
]

# Display diagnostics for categorical features
print("\n ======================== DIAGNOSTIC OF CATEGORICAL FEATURES ========================\n")
build_categorical_diagnostics(df_prepared, categorical_features)


 ======================== DIAGNOSTIC OF CATEGORICAL FEATURES ========================



,feature,nulls_%,unique_categories,top_category_pct,rare_categories_pct,target_median_variance,signal_suggestion,null_treatment_suggestion,encoding_suggestion
0,neighbourhood_cleansed,0.000000,16,0.488765,0.009790,0.133178,useful_signal,no_impute,frequency
1,room_type,0.000000,4,0.687599,0.002191,0.788047,useful_signal,no_impute,onehot
2,property_group,0.000000,6,0.772308,0.009977,0.080257,useful_signal,no_impute,onehot
3,property_group_room,0.000000,19,0.629744,0.023077,0.553901,useful_signal,no_impute,frequency
4,minimum_nights_segment,0.000000,3,0.898601,0.000000,0.036694,weak_signal,no_impute,onehot
5,host_verifications_grouped,0.000000,4,0.777716,0.000559,0.022870,weak_signal,no_impute,onehot
6,review_scores_mean_segment,0.115897,3,0.377927,0.000000,0.017036,weak_signal,missing_category,onehot
7,host_total_listings_segment,0.000093,4,0.278674,0.000000,0.023860,weak_signal,mode,onehot
8,log_price,0.000000,3796,0.004569,1.000000,0.913116,useful_signal,no_impute,frequency/group_rare


El diagnóstico categórico permitirá evaluar la calidad, complejidad y potencial predictivo de las variables categóricas antes de definir su estrategia de transformación y codificación para el modelo.

En términos generales, el conjunto categórico presenta:

- Variables sin valores nulos
- Cardinalidades controladas
- Poca presencia de categorías raras
- Variables con señal útil clara hacia el target
- Variables adecuadas para modelos lineales tras encoding

No se observan variables categóricas extremadamente problemáticas ni cardinalidades excesivas.

**1. `neighbourhood_cleansed`**

La variable presenta 16 categorías, una dominancia moderada de la categoría principal (48.9%) y muy pocas categorías raras (~1%). Su target_median_variance de 0.133 indica que la ubicación contiene señal predictiva útil para explicar diferencias en el precio entre zonas. Debido a su cardinalidad moderada, utilizar *one-hot encoding* podría generar demasiadas columnas y aumentar la dimensionalidad del modelo  por lo que la sugerencia de *frequency encoding* representa una alternativa más eficiente y estable.

**2. `room_type`**

La variable presenta únicamente 4 categorías y una señal predictiva muy fuerte (target_median_variance = 0.788), indicando que el tipo de habitación tiene un impacto significativo sobre el precio. Aunque existe una categoría dominante (68.7%), la baja cardinalidad sugiere aplicar *one-hot encoding* sin problemas de dimensionalidad. Esta variable probablemente será una de las features categóricas más importantes del modelo.

**3. `property_group`**

La variable contiene 6 categorías y una categoría dominante importante (77.2%). Su target_median_variance de 0.080 sugiere una señal útil moderada respecto al target. Debido a su baja cardinalidad y estructura relativamente simple, *one-hot encoding* se sugiere como una estrategia adecuada. Sin embargo, podría existir cierta redundancia con variables como `room_type` y `property_group_room`.

4. `property_group_room`

La variable presenta 19 categorías, una señal predictiva fuerte (target_median_variance = 0.553) y baja proporción de categorías raras (~2.3%). La combinación entre tipo de propiedad y tipo de habitación parece capturar relaciones más específicas relacionadas con el precio del listing. Debido a su cardinalidad moderada, *frequency encoding* se sugiere que puede conservar señal predictiva evitando una expansión excesiva de columnas.

5. `minimum_nights_segment`

La variable contiene únicamente 3 categorías, aunque presenta una fuerte dominancia de la categoría principal (89.8%). Su target_median_variance de 0.036 indica señal relativamente débil, sin embargo, al provenir de un proceso de binning sobre `minimum_nights`, puede ayudar a representar comportamientos no lineales difíciles de capturar mediante la variable numérica continua original. Debido a su baja cardinalidad, *one-hot encoding* es apropiado.

6. `host_verifications_grouped`

La variable  presenta 4 categorías, alta dominancia de la categoría principal (77.8%) y señal limitada respecto al target (target_median_variance = 0.023). Aunque probablemente no será una feature dominante dentro del modelo, puede aportar información secundaria relacionada con el nivel de confianza o profesionalización del host. Debido a su baja cardinalidad, *one-hot encoding* es una estrategia adecuada.

7. `review_scores_mean_segment`

La variable contiene 3 categorías (*low_review, medium_review, high_review*) y un ~11.6% de valores nulos, correspondientes a propiedades sin reseñas. La categoría dominante representa ~37.8%, lo que indica una distribución relativamente equilibrada entre segmentos. Su señal predictiva es débil (target_median_variance = 0.017), pero la segmentación aporta valor al diferenciar propiedades según niveles de calidad percibida, algo que la variable continua original no capturaba con claridad debido a la saturación en valores altos. La sugerencia de tratamiento es asignar una categoría explícita para los nulos (missing_category) y aplicar one-hot encoding para preservar la interpretabilidad.

8. `host_total_listings_segment`

La variable presenta 4 categorías y prácticamente no tiene valores nulos (~0.01%). La distribución es balanceada, con cada segmento representando entre 22% y 28% de los casos. Su señal predictiva es limitada (target_median_variance = 0.024), pero refleja un aspecto estructural del mercado, la escala de operación del anfitrión. La sugerencia de tratamiento es imputar los pocos nulos con la moda y aplicar one-hot encoding para capturar las diferencias entre segmentos.

### Diagnóstico de variables binarias

Las variables booleanas representan características binarias que indican la presencia o ausencia de determinados atributos dentro del listing. Aunque este tipo de variables suele presentar una estructura más simple que las variables numéricas o categóricas, su evaluación sigue siendo importante para determinar su utilidad predictiva dentro del modelo.

En esta sección se analizarán las variables booleanas candidatas considerando aspectos como:

- Distribución entre valores `True` y `False`
- Posibles desbalances
- Diferencias del target entre ambos grupos
- Relación lineal con el target

El objetivo es identificar qué variables binarias contienen señal útil para el modelo y detectar posibles features con baja variabilidad o capacidad explicativa limitada.

In [63]:
# Import diagnostic function
from src.features.analysis.diagnostics.diagnose_features import build_boolean_diagnostics

# Define boolean features to analyze (including target)
boolean_features = [
    'instant_bookable', 'has_review', 'host_is_superhost',
    'has_washer', 'has_pool', 'has_streaming_platform',
    'has_wifi', 'has_kitchen', 'has_hot_water',
    'has_essentials', 'has_bed_linens', 'has_microwave', 'has_refrigerator',
    'has_air_conditioning', 'has_heating', 'has_cooking_basics',
    'has_dishes_and_silverware', 'has_iron', 'has_hair_dryer',
    'has_dedicated_workspace', 'has_dining_table', 'has_dishwasher',
    'has_freezer', 'has_coffee_maker', 'has_blender', 'has_self_check_in',
    'has_private_entrance', 'has_elevator', 'has_free_parking',
    'has_pets_allowed', 'has_cleaning_available_during_stay', 'has_tv',
    'has_pool_table', 'has_piano', 'has_game_console',
    'has_ping_pong_table', 'has_patio_or_balcony', 'has_backyard',
    'has_sauna', 'has_city_skyline_view', 'has_outdoor_furniture',
    'has_outdoor_dining_area', 'has_smoke_alarm',
    'has_carbon_monoxide_alarm', 'has_fire_extinguisher',
    'has_exterior_security_cameras_on_property', 'has_first_aid_kit',
    # target
    'log_price']

# Display diagnostics for boolean features
print("\n ======================== DIAGNOSTIC OF BOOLEAN FEATURES ========================\n")
build_boolean_diagnostics(df_prepared, boolean_features)


 ======================== DIAGNOSTIC OF BOOLEAN FEATURES ========================



,feature,nulls_%,true_pct,target_median_diff,corr_with_target,signal_suggestion,null_treatment_suggestion,keep_suggestion
0,instant_bookable,0.000000,0.457389,0.175186,0.110405,useful_signal,no_impute,drop_candidate
1,has_review,0.000000,0.884103,0.144063,-0.071128,weak_signal,no_impute,drop_candidate
2,host_is_superhost,0.067879,0.448535,0.142964,0.109071,useful_signal,evaluate_missingness,drop_candidate
3,has_washer,0.000000,0.572914,0.324008,0.204226,strong_signal,no_impute,keep_candidate
4,has_pool,0.000000,0.074825,0.458954,0.166555,strong_signal,no_impute,keep_candidate
5,has_streaming_platform,0.000000,0.136783,0.153733,0.091871,useful_signal,no_impute,drop_candidate
6,has_wifi,0.000000,0.990443,0.089612,-0.000781,weak_signal,no_impute,drop_candidate
7,has_kitchen,0.000000,0.892075,0.317760,0.085740,strong_signal,no_impute,drop_candidate
8,has_hot_water,0.000000,0.844009,0.275750,0.096320,useful_signal,no_impute,drop_candidate
9,has_essentials,0.000000,0.716876,0.252085,0.139927,useful_signal,no_impute,evaluate


El análisis de variables booleanas tuvo como objetivo evaluar qué características binarias aportan señal predictiva útil para el modelo y cuáles presentan baja capacidad explicativa o redundancia respecto a otras variables derivadas.

Se analizaron tanto variables booleanas originales del dataset como las variables `has_amenity` generadas durante el proceso de feature engineering.

Las principales métricas utilizadas fueron:

- **true_pct**: proporción de observaciones donde la variable toma el valor *True*.
- **target_median_diff**: diferencia absoluta entre la mediana del target para los grupos *True* y *False*.
- **corr_with_target**: correlación entre la variable binaria y el target.
- **signal_suggestion**: evaluación heurística de la fuerza de señal de la variable.
- **keep_suggestion**: recomendación preliminar sobre si la variable debería mantenerse, evaluarse o descartarse para el modelo final.

Las variables booleanas `instant_bookable`, `has_review` y `host_is_superhost` presentan señal moderada, pero no suficientemente fuerte bajo los criterios definidos para selección final de features binarias.

- `instant_bookable`

La variable presenta una distribución relativamente balanceada (45.7% de valores *True*) y señal útil moderada (target_median_diff = 0.175, corr_with_target = 0.110). Aunque parece capturar cierto nivel de profesionalización y facilidad de reserva del listing, su capacidad discriminativa resulta limitada en comparación con otras variables más relevantes, por lo que fue clasificada como *drop_candidate*.

- `has_review`

La variable presenta una distribución altamente desbalanceada (88.4% de valores *True*) y señal relativamente débil (corr_with_target = -0.071). Debido a que la mayoría de listings ya poseen reviews, la variable aporta poca capacidad de separación entre observaciones. Además, puede reflejar factores temporales más que atributos reales del listing.

- `host_is_superhost`

Aunque la variable muestra una señal moderadamente útil (target_median_diff = 0.145, corr_with_target = 0.113) y posee interpretación de negocio clara relacionada con reputación y experiencia del host, su aporte marginal parece limitado frente a otras variables con mayor poder discriminativo.

**Intepretación de variables `has_amenity`**

Las variables `has_amenity` representan amenities específicas transformadas a variables binarias. El objetivo de estas variables es identificar amenities altamente discriminativas que aporten información adicional relevante, especialmente aquellas asociadas con lujo, confort, profesionalización o diferenciación económica del listing.

Las siguientes amenities muestran señal fuerte y representan candidatas sólidas para permanecer como variables individuales del modelo:

- `has_tv`: Presenta la mayor diferencia de medianas (0.70) y fuerte correlación (0.29). Parece capturar listings más completos o premium.
- `has_hair_dryer`: La mayor correlación observada (0.30) y alta diferencia del target (0.47). Amenity altamente discriminativa.
- `has_free_parking`: Señal fuerte y buena interpretabilidad económica, asociada probablemente con listings de mayor tamaño o mejor ubicación.
- `has_elevator`: Amenity relacionada con edificios de mayor nivel y propiedades urbanas premium.
- `has_air_conditioning`: Aunque relativamente poco frecuente (9%), presenta señal muy fuerte y clara asociación con listings de mayor precio.
- `has_pool`: Amenity altamente distintiva y asociada claramente con listings premium o vacacionales.
- `has_self_check_in`: Probablemente refleja profesionalización y experiencia optimizada del host.
- `has_coffee_maker`: Señal fuerte y posible asociación con listings más equipados y orientados a estadías cómodas.
- `has_outdoor_furniture`: Amenity relacionada con espacios exteriores y propiedades de mayor atractivo.
- `has_washer`: Buena correlación y fuerte separación del target, posiblemente asociada con estancias más completas o de mayor duración.

Algunas variables muestran señal moderada y podrían evaluarse posteriormente antes o durante el entrenamiento del modelo:

- `has_essentials`
- `has_bed_linens`
- `has_dining_table`
- `has_freezer`
- `has_blender`
- `has_patio_or_balcony`
- `has_city_skyline_view`
- `has_smoke_alarm`

Varias amenities fueron clasificadas como candidatas a exclusión debido a alguno de los siguientes factores:

- señal extremadamente débil
- correlaciones cercanas a cero
- alta redundancia
- distribución extremadamente desbalanceada

Ejemplos claros incluyen:

- `has_wifi`
- `has_game_console`
- `has_exterior_security_cameras_on_property`
- `has_first_aid_kit`
- `has_piano`

En muchos casos, estas amenities están presentes en casi todos los listings, son extremadamente raras o no muestran diferencias relevantes respecto al target. Por ello, su aporte marginal al modelo probablemente sea limitado.

El diagnóstico booleano permitirá reducir considerablemente el conjunto inicial de amenities binarias, facilitando un proceso de selección más enfocado y manejable. En lugar de incluir decenas de variables `has_amenity`, el objetivo es conservar únicamente amenities individuales que aporten señal específica adicional respecto a `amenity_score` y `amenity_count`. En general, las amenities relacionadas con confort, lujo, equipamiento premium o experiencia del huésped muestran mayor capacidad discriminativa y representan las candidatas más prometedoras para el modelo final.

### Implementación de técnicas de preprocesamiento

Una vez construidas las variables derivadas y realizado el diagnóstico de las features numéricas, categóricas y booleanas, se procederá a definir y aplicar las técnicas de preprocesamiento necesarias para preparar el dataset final de modelado.

El objetivo de esta etapa es convertir las variables a representaciones más adecuadas para el entrenamiento de modelos de machine learning, mejorando aspectos como:

- estabilidad estadística
- manejo de distribuciones sesgadas
- tratamiento de valores faltantes
- codificación de variables categóricas
- escalamiento de variables numéricas

Para ello, primero se evaluaron las características de cada tipo de variable con el fin de determinar las transformaciones más apropiadas según su distribución, cardinalidad, señal predictiva y comportamiento respecto al target. Posteriormente, todas las transformaciones seleccionadas fueron integradas dentro de un pipeline modular de preprocesamiento, permitiendo aplicar el mismo flujo de procesamiento de forma consistente y reproducible tanto en entrenamiento como en inferencia.

Las funciones utilizadas en esta sección fueron previamente modularizadas dentro de la carpeta `src/preprocess` del proyecto, lo que permite mantener una organización clara y reutilizable. Este enfoque no solo mejora la calidad de las features, sino que también asegura que el proceso pueda escalar y aplicarse de forma consistente en entornos de entrenamiento e inferencia.

In [64]:
df_prepared.info()

<class 'pandas.DataFrame'>
RangeIndex: 21450 entries, 0 to 21449
Data columns (total 83 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   neighbourhood_cleansed                        21450 non-null  str    
 1   latitude                                      21450 non-null  float64
 2   longitude                                     21450 non-null  float64
 3   property_type                                 21450 non-null  str    
 4   room_type                                     21450 non-null  str    
 5   accommodates                                  21450 non-null  int64  
 6   bathrooms                                     21438 non-null  float64
 7   bedrooms                                      21353 non-null  float64
 8   beds                                          21434 non-null  float64
 9   minimum_nights                                21450 non-null  int64  
 1

In [67]:
# Import preprocessing pipeline
from src.preprocess.preprocess_features import (
    fit_preprocessing_pipeline,
    transform_preprocessing_pipeline
)

# Fit preprocessing components using the full dataset
# This step is only used for feature evaluation and
# feature selection purposes in Notebook 02
preprocessors = fit_preprocessing_pipeline(
    df_prepared
)

# Apply the complete preprocessing pipeline
df_prepared = transform_preprocessing_pipeline(
    df_prepared,
    preprocessors
)

df_prepared.info()

<class 'pandas.DataFrame'>
RangeIndex: 21450 entries, 0 to 21449
Data columns (total 91 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   latitude                                          21450 non-null  float64
 1   longitude                                         21450 non-null  float64
 2   property_type                                     21450 non-null  str    
 3   minimum_nights                                    21450 non-null  int64  
 4   amenities                                         21450 non-null  object 
 5   host_is_superhost                                 21450 non-null  float64
 6   host_verifications                                21450 non-null  object 
 7   host_total_listings_count                         21450 non-null  float64
 8   instant_bookable                                  21450 non-null  float64
 9   review_scores_rating        

In [68]:
# Display the first rows of the prepared dataset
df_prepared.head()

,latitude,longitude,property_type,minimum_nights,amenities,host_is_superhost,host_verifications,host_total_listings_count,instant_bookable,review_scores_rating,...,room_type_entire_home/apt,room_type_hotel_room,room_type_private_room,room_type_shared_room,property_group_apartment,property_group_guesthouse,property_group_hotel,property_group_house,property_group_other,property_group_unique/nature
0,19.38283,-99.27178,Entire villa,1,"[Garden view, Resort access, Washer, Courtyard...",0.0,"[email, phone, work_email]",1.0,0.0,NaN,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,19.41162,-99.17794,Entire home,1,"[Piano, Patio or balcony, Wifi, Refrigerator, ...",0.0,"[email, phone, work_email]",13.0,0.0,4.59,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,19.43977,-99.15605,Entire condo,15,"[Self check-in, Dining table, Elevator, Wifi, ...",1.0,"[email, phone]",5.0,0.0,4.87,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,19.40826,-99.18659,Entire home,2,"[Pack ’n play/Travel crib, Self check-in, Heat...",1.0,"[email, phone]",6.0,0.0,4.88,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,19.39675,-99.17581,Private room in rental unit,4,"[Elevator, Wifi, Hot water, Microwave, Refrige...",1.0,"[email, phone]",3.0,0.0,4.84,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


## 6. Evaluación y Selección Final de Variables

Después del proceso de creación y preprocesamiento de variables, se realizará una etapa final de evaluación de características con el objetivo de construir un conjunto de variables más robusto, compacto y relevante para el modelado. En esta fase se analizarab las variables desde una perspectiva multivariable, considerando no solo su relación individual con la variable objetivo, sino también posibles redundancias, multicolinealidad y aporte predictivo conjunto.

Para ello se evaluarán distintos criterios estadísticos y de modelado, incluyendo análisis de correlación entre variables, detección de multicolinealidad, métricas de dependencia no lineal e importancia de variables basada en modelos. 

El objetivo de esta etapa no será maximizar la cantidad de variables, sino seleccionar un conjunto de características con señal útil, interpretabilidad y capacidad de generalización para el modelo final. Para realizar esta evaluación y selección final de variables crearemos el conjunto de datos `df_eval`.

In [69]:
df_eval = df_prepared[[

    # Numeric features
    'calculated_host_listings_count_entire_homes_log',
    'calculated_host_listings_count_private_rooms_log',
    'dist_to_nearest_attraction_log', 'beds_log', 'amenity_score',
    'accommodates_log', 'bedrooms_log', 'bathrooms_log', 'amenity_count',
    'attractions_within_radius_log', 'commercial_within_radius',


    # Categorical features
    'room_type_entire_home/apt','room_type_hotel_room', 'room_type_private_room','room_type_shared_room',
    'property_group_apartment','property_group_guesthouse', 'property_group_hotel',
    'property_group_house', 'property_group_other','property_group_unique/nature',
    'neighbourhood_cleansed_freq', 'property_group_room_freq',
    'host_total_listings_segment_ord', 'review_scores_mean_segment_ord', 
    'minimum_nights_segment_ord', 'host_verifications_grouped_ord',


    # Boolean features
    'host_is_superhost','instant_bookable', 'has_review',
    'has_tv', 'has_elevator', 'has_free_parking', 'has_coffee_maker', 
    'has_outdoor_furniture', 'has_air_conditioning','has_self_check_in', 'has_pool',

    # Target
    'log_price'

    ]].copy()

df_eval.head()

,calculated_host_listings_count_entire_homes_log,calculated_host_listings_count_private_rooms_log,dist_to_nearest_attraction_log,beds_log,amenity_score,accommodates_log,bedrooms_log,bathrooms_log,amenity_count,attractions_within_radius_log,...,has_review,has_tv,has_elevator,has_free_parking,has_coffee_maker,has_outdoor_furniture,has_air_conditioning,has_self_check_in,has_pool,log_price
0,-0.269577,0.0,1.142523,0.000000,0.143200,-0.652946,-0.563382,-0.573426,-1.361130,-1.147290,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,8.208764
1,0.339627,1.0,-0.314243,3.709511,0.557312,3.126331,2.973722,3.787270,-0.460081,0.737396,...,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,9.798127
2,-0.269577,0.0,-0.360812,0.000000,0.521123,-0.652946,-0.563382,-0.573426,-0.331360,0.476090,...,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,6.381816
3,0.460845,0.0,0.239169,4.204426,0.673100,3.420238,2.973722,3.491134,0.891493,-0.335600,...,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,8.208764
4,-0.730423,1.0,0.652661,0.000000,0.551671,-0.652946,-0.563382,-0.573426,-0.588802,-1.147290,...,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,5.771441


### Redundancia / Correlación entre características

Se realizará un análisis de redundancia y correlación entre características con el objetivo de identificar posibles relaciones fuertes entre variables que pudieran representar información similar o duplicada. Este análisis permitirá detectar grupos de variables altamente correlacionadas, evaluar posibles problemas de multicolinealidad y entender mejor la relación estructural entre las características creadas y transformadas durante el proceso de feature engineering.

La presencia de correlaciones elevadas no implica necesariamente que una variable deba eliminarse automáticamente. Sin embargo, sí proporciona evidencia útil para identificar variables redundantes, simplificar el modelo y mejorar la interpretabilidad y estabilidad del proceso de entrenamiento.

En esta etapa se analizaron tanto variables numéricas como variables binarias y ordinales ya transformadas al espacio numérico final del modelo.

In [70]:
from src.features.analysis.selection.correlation_analysis import feature_correlation_analysis

corr_matrix, high_corr_pairs = feature_correlation_analysis(
    df_eval,
    target="log_price",
    threshold=0.75
)

print("================== Feature Correlation Analysis (Threshold = 0.75) ==================")
high_corr_pairs.sort_values(by='correlation', ascending=False)

================== Feature Correlation Analysis (Threshold = 0.75) ==================


,feature_1,feature_2,correlation
2,amenity_score,amenity_count,0.883704
6,room_type_entire_home/apt,property_group_room_freq,0.845794
1,beds_log,accommodates_log,0.794495
3,attractions_within_radius_log,commercial_within_radius,0.774555
4,commercial_within_radius,neighbourhood_cleansed_freq,0.773424
8,property_group_apartment,property_group_room_freq,0.771046
0,dist_to_nearest_attraction_log,attractions_within_radius_log,-0.788564
7,room_type_private_room,property_group_room_freq,-0.810174
5,room_type_entire_home/apt,room_type_private_room,-0.969939


El análisis de correlación entre variables permitió identificar posibles relaciones de redundancia entre características candidatas para el modelo. Para este análisis se utilizó un umbral absoluto de correlación de 0.75, considerando tanto relaciones positivas como negativas.

En general, los resultados muestran que la mayoría de las variables mantienen niveles de correlación aceptables, lo cual indica que el conjunto final de características conserva una buena diversidad de información. Sin embargo, se identificaron algunos grupos de variables con relaciones fuertes que requieren evaluación adicional antes de la selección final.

**1. Amenity Score vs Amenity Count**

La correlación positiva más alta encontrada fue entre `amenity_score` y `amenity_count` (0.88), lo cual era esperado debido a que ambas variables representan el nivel general de equipamiento del alojamiento.

No obstante, ambas variables capturan conceptos ligeramente distintos:

- `amenity_count` representa únicamente la cantidad total de amenidades disponibles.
- `amenity_score` incorpora una ponderación basada en la relevancia o calidad relativa de ciertas amenidades.

Debido a esta fuerte redundancia, mantener ambas variables podría introducir información duplicada en el modelo. Como posible acción, se podría conservar únicamente `amenity_score`, ya que aporta una representación más diversa y contextual del equipamiento del listing.

**2. Room Type vs Property Group Room Frequency**

Se observa una correlación fuerte entre:

- `room_type_entire_home/apt`
- `room_type_private_room`
- `property_group_room_freq`

Esto sugiere que la variable de frecuencia `property_group_room_freq` está capturando parcialmente información relacionada con el tipo de habitación del alojamiento.

En particular:

- valores altos de `property_group_room_freq` parecen asociarse principalmente con propiedades tipo `Entire home/apartment`
- mientras que valores bajos se relacionan con `Private room`

Esto indica una posible redundancia conceptual entre ambas representaciones. Se podría evaluar si `property_group_room_freq` realmente aporta información adicional al modelo más allá de `room_type`. En caso contrario, podría eliminarse para reducir colinealidad e incrementar interpretabilidad.

**3. Beds vs Accommodates**

La correlación entre `beds_log` y `accommodates_log` (0.79) refleja una relación estructural esperada dentro del dataset: alojamientos con mayor capacidad suelen incluir un mayor número de camas.

Aunque existe redundancia parcial, ambas variables representan conceptos distintos:

- `accommodates` representa la capacidad máxima de huéspedes
- `beds` representa la infraestructura física disponible

Debido a que ambas variables pueden aportar información complementaria, no se considera eliminar automáticamente ninguna de ellas en esta etapa. Su permanencia final deberá evaluarse posteriormente mediante importancia de variables o métricas de multicolinealidad.

**4. Variables Geoespaciales**

Las variables:

- `dist_to_nearest_attraction_log`
- `attractions_within_radius_log`
- `commercial_within_radius`
- `neighbourhood_cleansed_freq`

Mostraron correlaciones relativamente altas entre sí. Esto sugiere que todas están capturando distintas dimensiones de densidad urbana, accesibilidad y centralidad geográfica.

Por ejemplo:

- menores distancias a atracciones suelen asociarse con una mayor cantidad de comercios y puntos de interés cercanos.
- ciertas colonias (`neighbourhood_cleansed_freq`) concentran naturalmente mayor actividad comercial y turística.

A pesar de esta relación, cada variable describe el entorno desde una perspectiva distinta, por lo que no se consideran redundantes de forma estricta. En consecuencia, se pueden mantener y evaluar posteriormente su contribución mediante modelos basados en importancia de variables.

**5. Correlación Negativa entre Variables One-Hot**

La correlación negativa extrema entre `room_type_entire_home/apt` y  `room_type_private_room` es completamente esperada (0.97) debido al proceso de one-hot encoding. Este comportamiento no representa un problema de multicolinealidad real, sino una consecuencia natural de variables mutuamente excluyentes derivadas de una misma categoría original. Por esta razón, este tipo de correlaciones no se consideran criterio suficiente para eliminar variables codificadas mediante one-hot encoding.

### Análisis de Multicolinealidad (VIF)

Después del análisis de correlación entre variables, se realizará una evaluación de multicolinealidad utilizando el indicador Variance Inflation Factor (VIF). A diferencia del análisis de correlación realizado previamente, cuyo objetivo fue identificar posibles relaciones de redundancia o similitud conceptual entre variables, el análisis VIF se enfoca específicamente en medir multicolinealidad lineal entre predictores.

Mientras que la correlación permite detectar variables que podrían estar capturando información similar, el VIF evalúa cuánto puede explicarse estadísticamente una variable a partir del resto de características del modelo, ayudando a identificar posibles problemas de estabilidad e interpretación en el proceso de entrenamiento. En otras palabras, evalúa el nivel de redundancia estructural entre predictores y ayuda a identificar variables que podrían aportar información duplicada al modelo.

Valores elevados de VIF sugieren que una variable comparte una gran cantidad de información con otras características, lo cual puede generar problemas de estabilidad e interpretabilidad, especialmente en modelos lineales. Para este análisis se consideraran únicamente variables numéricas, ordinales y binarias transformadas al espacio numérico final del modelo. Las variables derivadas de one-hot encoding serán excluidas debido a que su naturaleza mutuamente excluyente puede inflar artificialmente el VIF sin representar necesariamente un problema real de modelado.

Como referencia general, se utilizaran los siguientes criterios de interpretación:

- **VIF < 5**: multicolinealidad baja o aceptable.
- **VIF entre 5 y 10**: posible redundancia moderada, requiere evaluación.
- **VIF > 10**: multicolinealidad alta, posible redundancia fuerte.

In [71]:
df_eval.info()

<class 'pandas.DataFrame'>
RangeIndex: 21450 entries, 0 to 21449
Data columns (total 39 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   calculated_host_listings_count_entire_homes_log   21450 non-null  float64
 1   calculated_host_listings_count_private_rooms_log  21450 non-null  float64
 2   dist_to_nearest_attraction_log                    21450 non-null  float64
 3   beds_log                                          21450 non-null  float64
 4   amenity_score                                     21450 non-null  float64
 5   accommodates_log                                  21450 non-null  float64
 6   bedrooms_log                                      21450 non-null  float64
 7   bathrooms_log                                     21450 non-null  float64
 8   amenity_count                                     21450 non-null  float64
 9   attractions_within_radius_lo

In [72]:
from src.features.analysis.selection.vif_analysis import vif_analysis

# Variables a excluir del análisis VIF
excluded_vif_features = [

    # One-hot encoded variables
    'room_type_entire_home/apt',
    'room_type_hotel_room',
    'room_type_private_room',
    'room_type_shared_room',

    'property_group_apartment',
    'property_group_guesthouse',
    'property_group_hotel',
    'property_group_house',
    'property_group_other',
    'property_group_unique/nature',

    # Target
    'log_price'
]

# Dataset para VIF
df_vif = df_eval.drop(
    columns=excluded_vif_features
)

# Run VIF analysis
vif_results = vif_analysis(df_vif)

print("================== Variance Inflation Factor Analysis ==================")
vif_results

================== Variance Inflation Factor Analysis ==================


,feature,VIF
0,amenity_score,69.681854
1,host_verifications_grouped_ord,15.287557
2,has_review,14.361082
3,has_tv,12.176282
4,host_total_listings_segment_ord,8.968745
5,has_coffee_maker,7.753787
6,beds_log,4.742736
7,amenity_count,4.106806
8,attractions_within_radius_log,3.788299
9,calculated_host_listings_count_private_rooms_log,3.777475


El análisis de Variance Inflation Factor (VIF) permitió evaluar el nivel de multicolinealidad entre las variables candidatas del modelo. En general, la mayoría de las características presentaron valores de VIF bajos o moderados, lo cual indica una estructura relativamente estable y con niveles aceptables de dependencia lineal.

Sin embargo, algunas variables mostraron niveles elevados de VIF que sugieren posibles relaciones de redundancia estructural y requieren evaluación adicional antes de la selección final del modelo.

**1. Amenity Score**

La variable `amenity_score` presentó el valor de VIF más alto (69.94), indicando una multicolinealidad extremadamente elevada respecto al resto de variables del modelo. Este resultado podría ser esperado debido a que `amenity_score` fue construido a partir de múltiples amenidades individuales, algunas de las cuales también permanecen como variables independientes dentro del conjunto final. En consecuencia, gran parte de la información contenida en el score puede explicarse mediante las variables binarias asociadas a amenidades específicas.

Adicionalmente, `amenity_score` mostró previamente una correlación muy alta con `amenity_count`, reforzando la evidencia de redundancia dentro del grupo de variables relacionadas con equipamiento del listing.

Como posible acción, considero dos posibles enfoques:

- conservar `amenity_score` y reducir el número de amenidades individuales;
- conservar únicamente las amenidades individuales más relevantes y eliminar el score agregado.

Debido a que `amenity_score` captura una representación global más rica del equipamiento del alojamiento, mantener el score y limitar las amenidades individuales podría ser alternativa más sólida e interpretable.

**2. Host Verifications Grouped Ordinal**

La variable `host_verifications_grouped_ordinal` presentó un VIF elevado (15.25), sugiriendo una dependencia importante con otras variables relacionadas al nivel de profesionalización o confiabilidad del host.

Es probable que esta variable comparta información con características como:

- `host_is_superhost`
- `has_review`
- `instant_bookable`
- `host_total_listings_segment_ordinal`

Ya que todas representan distintos indicadores indirectos de experiencia, reputación o madurez del anfitrión dentro de la plataforma. A pesar de esta relación, la variable sigue representando un concepto interpretable y potencialmente útil. Por ello, no se considera eliminar automáticamente en esta etapa, aunque sí evaluar posteriormente su importancia real dentro del modelo final.

**3. Has Review**

La variable `has_review` también mostró un VIF elevado (14.37).

Este comportamiento probablemente se debe a que la presencia de reseñas está fuertemente asociada con otras características relacionadas con antigüedad, actividad y reputación del listing o del host. No obstante, desde una perspectiva conceptual, `has_review` aporta información relativamente limitada debido a su baja variabilidad y a que la mayoría de los listings ya poseen reseñas.

Considerando además que en análisis previos presentó señal predictiva relativamente débil, esta variable surge como una posible candidata a eliminación en la selección final.


**4. Has TV y Has Coffee Maker**

Las variables `has_tv` (12.18) y `has_coffee_maker` (7.75) también mostraron niveles relativamente altos de VIF. Esto sugiere que ambas amenidades están funcionando parcialmente como proxies generales de listings mejor equipados, más que como factores independientes de precio.

En otras palabras, estas variables probablemente están absorbiendo parte de la información ya contenida dentro de `amenity_score`. Debido a ello, mantener simultáneamente demasiadas amenidades individuales junto con un score agregado puede introducir redundancia innecesaria dentro del modelo.

**5. Variables con VIF Moderado**

Variables como:

- `beds_log`
- `accommodates_log`
- `commercial_within_radius`
- `attractions_within_radius_log`

Presentaron niveles moderados de VIF (3–5), lo cual refleja relaciones estructurales esperadas dentro del dataset. Por ejemplo, alojamientos con mayor capacidad suelen tener más camas y zonas con mayor densidad comercial suelen presentar más atracciones cercanas. Sin embargo, los niveles observados no indican problemas severos de multicolinealidad y las variables continúan representando dimensiones distintas del problema.

**6. Variables con VIF Bajo**

La mayoría de las variables restantes presentaron valores de VIF bajos (< 3), indicando una contribución relativamente independiente dentro del conjunto final de características Esto sugiere que el proceso de selección y reducción de variables realizado previamente permitió construir un espacio de features relativamente compacto y con niveles controlados de redundancia lineal.

### Análisis de dependencia estadística (Mutual Information)

Después de evaluar redundancia y multicolinealidad entre variables, se realizará un análisis de Mutual Information (métrica de la teoría de la información que cuantifica cuánta dependencia estadística existe entre dos variables) con el objetivo de medir la capacidad informativa de cada característica respecto a la variable objetivo.

A diferencia de la correlación tradicional, Mutual Information no se limita a relaciones lineales. Esta métrica permite detectar dependencias tanto lineales como no lineales entre variables, proporcionando una evaluación más flexible del poder predictivo potencial de cada feature. Mientras que los análisis previos se enfocaron en estudiar relaciones entre predictores, el análisis de Mutual Information se centra específicamente en evaluar cuánto conocimiento aporta cada variable sobre el comportamiento del target.

Valores altos de Mutual Information indican que una característica reduce significativamente la incertidumbre sobre la variable objetivo, sugiriendo una mayor relevancia predictiva dentro del modelo. En esta etapa se evaluaron todas las variables transformadas al espacio numérico final del modelo, incluyendo variables numéricas, binarias, ordinales y codificadas mediante one-hot y frequency encoding.

In [73]:
from src.features.analysis.selection.mutual_info_analysis import mutual_information_analysis

# Dataset for MI analysis
df_mi = df_eval.copy()

# Run analysis
mi_results = mutual_information_analysis(
    df_mi,
    target="log_price"
)

mi_results

,feature,mutual_information
0,calculated_host_listings_count_entire_homes_log,0.324134
1,calculated_host_listings_count_private_rooms_log,0.313282
2,property_group_room_freq,0.306704
3,amenity_score,0.306247
4,accommodates_log,0.304730
5,commercial_within_radius,0.240622
6,dist_to_nearest_attraction_log,0.234526
7,bedrooms_log,0.213260
8,bathrooms_log,0.212290
9,room_type_entire_home/apt,0.207430


El análisis de Mutual Information permitió evaluar la relevancia predictiva individual de cada característica respecto a la variable objetivo. A diferencia de la correlación tradicional, esta métrica captura tanto relaciones lineales como no lineales, proporcionando una visión más completa del valor informativo de cada variable.

En términos generales, los resultados muestran que gran parte de la capacidad predictiva del modelo proviene de variables relacionadas con la escala del alojamiento, el tipo de propiedad, la actividad del anfitrión, la ubicación y el nivel de equipamiento.

**1. Variables con Mayor Relevancia Predictiva**

Las variables con mayores valores de Mutual Information fueron:

- `calculated_host_listings_count_entire_homes_log`
- `calculated_host_listings_count_private_rooms_log`
- `property_group_room_freq`
- `amenity_score`
- `accommodates_log`

Estas características representan algunos de los factores más importantes para explicar el comportamiento del precio.

**2. Actividad y especialización del host**

Las dos variables más relevantes corresponden al número de propiedades administradas por el anfitrión. Esto sugiere que el nivel de profesionalización del host tiene una relación importante con el precio de los alojamientos. Hosts con múltiples propiedades suelen operar bajo estrategias diferentes a las de anfitriones ocasionales, afectando tanto el posicionamiento como la estructura de precios.

**3. Tipo de propiedad**

`property_group_room_freq` aparece como una de las variables más informativas del conjunto. Este resultado refuerza una conclusión observada previamente: el tipo de propiedad es uno de los factores estructurales más importantes para explicar diferencias de precio dentro del dataset.

**4. Nivel de equipamiento**

`amenity_score` se posiciona entre las variables más relevantes del modelo. Esto valida la decisión de construir una métrica agregada de amenidades, ya que demuestra una capacidad predictiva superior a la mayoría de las amenidades individuales consideradas por separado.

**5. Capacidad del alojamiento**

`accommodates_log`, junto con `bedrooms_log`, `bathrooms_log` y `beds_log`, confirma que el tamaño y capacidad de hospedaje constituyen uno de los principales determinantes del precio.

**6. Importancia de la Ubicación**

Las variables `commercial_within_radius`, `dist_to_nearest_attraction_log`, `attractions_within_radius_log`, `neighbourhood_cleansed_freq` obtuvieron niveles elevados de Mutual Information. Esto indica que la ubicación aporta una cantidad considerable de información al modelo, la combinación de variables geoespaciales parece capturar distintos aspectos de accesibilidad, densidad urbana, atractivo turístico y valor percibido de la zona.

**7. Room Type y Property Group**

Las variables derivadas de `room_type` presentan niveles de información relativamente altos. En particular `room_type_entire_home/apt` y `room_type_private_room` mantienen una contribución importante al modelo. Este resultado es consistente con el conocimiento de dominio, ya que el tipo de alojamiento suele generar diferencias significativas de precio.

Asimismo, algunas categorías específicas de `property_group`, como `apartment` y `house`, también muestran señal predictiva relevante.

**8. Amenidades Individuales**

Las amenidades individuales seleccionadas presentan niveles de Mutual Information moderados.

Entre las más relevantes destacan:

- `has_elevator`
- `has_coffee_maker`
- `has_tv`
- `has_self_check_in`
- `has_free_parking`

Sin embargo, ninguna de ellas supera el aporte de `amenity_score`. Este resultado refuerza la hipótesis observada durante el análisis de VIF: gran parte de la señal contenida en las amenidades individuales ya está siendo capturada por la variable agregada de equipamiento. Las amenidades individuales parecen aportar información complementaria y específica, más que convertirse en los principales determinantes del precio.

**9. Variables del Host**

Las variables relacionadas con reputación y características del anfitrión:

- `host_total_listings_segment_ordinal`
- `instant_bookable`
- `host_is_superhost`
- `has_review`
- `host_verifications_grouped_ordinal`

Presentan niveles de información relativamente bajos a moderados. Esto sugiere que, aunque contribuyen al modelo, su capacidad predictiva individual es inferior a la observada en variables de tamaño, ubicación o equipamiento.

**10. Variables con Señal Limitada**

Algunas categorías específicas presentan valores de Mutual Information cercanos a cero:

- `property_group_other`
- `room_type_hotel_room`
- `property_group_unique/nature`

Estos resultados indican que dichas variables aportan poca información adicional para explicar el precio dentro del conjunto analizado. Por tanto, surgen como posibles candidatas a eliminación durante la selección final de variables, especialmente si se busca simplificar el modelo sin perder capacidad predictiva significativa.

El análisis confirma que las variables más relevantes del modelo pueden agruparse en cinco grandes dimensiones:

1. Escala y profesionalización del anfitrión.
2. Tamaño y capacidad del alojamiento.
3. Tipo de propiedad y tipo de habitación.
4. Ubicación y accesibilidad.
5. Nivel general de equipamiento.

Asimismo, los resultados respaldan varias decisiones tomadas durante el proceso de feature engineering, especialmente la construcción de variables agregadas como `amenity_score`, así como la incorporación de variables geoespaciales y segmentaciones derivadas de características originales.

### Modelo de referencia para evaluación del dataset

Una vez completados los análisis individuales de relevancia, redundancia y multicolinealidad, se construyó un modelo de referencia con el objetivo de evaluar el comportamiento conjunto de las variables seleccionadas. A diferencia de los análisis anteriores, que estudian cada característica de forma aislada o mediante relaciones específicas entre variables, un modelo base nos permitirá observar cómo interactúan todas las features simultáneamente dentro de un entorno predictivo real.

En esta etapa se utilizó un modelo Random Forest Regressor debido a su capacidad para capturar relaciones no lineales, manejar interacciones complejas entre variables y proporcionar métricas de importancia de características que serán utilizadas posteriormente durante la selección final de variables.

El objetivo principal de este modelo no es obtener el mejor desempeño posible, sino establecer una referencia inicial del potencial predictivo del conjunto actual de variables y generar información adicional para apoyar las decisiones de selección de características.

Las métricas a evaluar serán:

- **R² (Coefficient of Determination):** proporción de variabilidad explicada por el modelo.
- **RMSE (Root Mean Squared Error):** error promedio penalizando errores grandes.
- **MAE (Mean Absolute Error):** error absoluto promedio de las predicciones.

In [74]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)
import numpy as np

# ================= BASELINE MODEL =================

# Features
X = df_eval.drop(
    columns=["log_price"]
)

# Target
y = df_eval["log_price"]

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Baseline model
rf_baseline = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Train
rf_baseline.fit(
    X_train,
    y_train
)

# Predictions
y_pred = rf_baseline.predict(X_test)

# ================= REAL PRICE METRICS =================

y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred)

real_rmse = root_mean_squared_error(
    y_test_real,
    y_pred_real
)

real_mae = mean_absolute_error(
    y_test_real,
    y_pred_real
)

# Metrics
results = {

    "R²": r2_score(
        y_test,
        y_pred
    ),

    "RMSE": root_mean_squared_error(
        y_test,
        y_pred
    ),

    "MAE": mean_absolute_error(
        y_test,
        y_pred
    )
}

print("\n================= BASELINE MODEL RESULTS =================")

print(f"R² (log_price): {results['R²']:.4f}")
print(f"RMSE (log_price): {results['RMSE']:.4f}")
print(f"MAE (log_price): {results['MAE']:.4f}")

print("\n================= REAL PRICE METRICS =================")

print(f"RMSE ($): ${real_rmse:,.2f}")
print(f"MAE ($): ${real_mae:,.2f}")


================= BASELINE MODEL RESULTS =================
R² (log_price): 0.7592
RMSE (log_price): 0.3557
MAE (log_price): 0.2483

================= REAL PRICE METRICS =================
RMSE ($): $1,448.39
MAE ($): $414.93


El modelo base obtuvo un desempeño sólido al utilizar el conjunto actual de variables seleccionadas, alcanzando un **R² de 0.7592**, lo que indica que aproximadamente el 75.9% de la variabilidad observada en los precios puede ser explicada por las características incluidas en el modelo.

Este resultado sugiere que las variables derivadas durante el proceso de feature engineering capturan una parte importante de los factores que influyen en el precio de los alojamientos, incluyendo aspectos relacionados con la capacidad del inmueble, la ubicación, el tipo de propiedad, las características del anfitrión y el nivel de equipamiento.

**1. Análisis de Error**

En escala logarítmica, el modelo obtuvo:

- **RMSE:** 0.3557
- **MAE:** 0.2483

Estas métricas indican que el error promedio de predicción es relativamente bajo considerando la variabilidad natural presente en los precios de alojamientos turísticos. Para facilitar la interpretación, las predicciones fueron transformadas nuevamente a la escala original de precios.

En términos monetarios, el modelo presenta:

- **RMSE:** \$1,448.39 MXN
- **MAE:** \$414.93 MXN

El MAE de aproximadamente $415 MXN implica que, en promedio, las predicciones difieren del precio real por alrededor de 413 unidades monetarias. Considerando la amplia dispersión de precios observada en el dataset, este nivel de error resulta razonable para una primera aproximación sin ajuste de hiperparámetros.

Por otro lado, el RMSE superior al MAE indica la presencia de algunos errores de mayor magnitud, algo esperado en problemas de precios donde existen propiedades significativamente más costosas que el promedio y cuya predicción suele ser más compleja.

**2. Evaluación General**

Los resultados obtenidos validan las decisiones tomadas durante las etapas anteriores de creación, transformación y selección preliminar de variables. En particular, sugieren que:

- Las variables geoespaciales aportan información relevante para explicar diferencias de precio.
- Las variables derivadas del tamaño y capacidad del alojamiento capturan una parte importante de la señal predictiva.
- Las características asociadas al anfitrión contribuyen a mejorar la capacidad explicativa del modelo.
- Las variables relacionadas con amenidades y equipamiento contienen información útil para la predicción.
- Las transformaciones aplicadas permiten representar adecuadamente relaciones que no son estrictamente lineales.

En conjunto, el desempeño alcanzado proporciona evidencia de que el conjunto actual de características posee una capacidad predictiva significativa y constituye una base sólida para continuar con la selección final de variables. Finalmente, el modelo entrenado será conservado para la siguiente etapa del análisis, donde se evaluará la importancia de cada característica dentro del Random Forest con el fin de identificar cuáles variables contribuyen realmente al desempeño predictivo cuando todas compiten simultáneamente dentro del modelo.

In [75]:
# Keep trained model for Feature Importante analysis
baseline_model = rf_baseline

### Importancia de Variables (Feature Importance)


Se realizó un análisis de importancia de variables utilizando el modelo Random Forest entrenado previamente. A diferencia de Mutual Information, que evalúa cada característica de forma independiente respecto al target, la importancia de variables permite medir la contribución real de cada feature cuando todas compiten simultáneamente dentro del modelo.

Este análisis resulta especialmente útil para identificar:

- Variables que aportan información única al modelo.
- Variables cuya importancia disminuye debido a la presencia de otras características correlacionadas.
- Variables con poca contribución predictiva que podrían ser candidatas a eliminación.
- Variables que concentran gran parte de la capacidad predictiva del modelo.

En Random Forest, la importancia se calcula a partir de la reducción acumulada de impureza generada por cada característica a lo largo de todos los árboles del ensamble. Cuanto mayor sea la reducción producida por una variable, mayor será su importancia relativa dentro del modelo.

Los resultados obtenidos servirán como uno de los criterios principales para definir el conjunto final de variables que será utilizado durante la etapa de modelado.

In [76]:
import pandas as pd

# ================= FEATURE IMPORTANCE ANALYSIS =================

feature_importance = pd.DataFrame({

    "feature": X_train.columns,
    "importance": baseline_model.feature_importances_

})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
).reset_index(drop=True)

print(
    "\n================= FEATURE IMPORTANCE ================="
)

display(feature_importance)


================= FEATURE IMPORTANCE =================


,feature,importance
0,bathrooms_log,0.177060
1,room_type_entire_home/apt,0.150847
2,accommodates_log,0.131125
3,dist_to_nearest_attraction_log,0.056142
4,commercial_within_radius,0.054604
5,amenity_score,0.053394
6,attractions_within_radius_log,0.042657
7,property_group_room_freq,0.037460
8,amenity_count,0.035582
9,bedrooms_log,0.029206


Interpretación del Análisis de Feature Importance

El análisis de importancia de variables permite identificar cuáles características son realmente utilizadas por el modelo para realizar sus predicciones cuando todas compiten simultáneamente dentro del proceso de aprendizaje.

A diferencia de Mutual Information, que evalúa cada variable de forma individual, Feature Importance refleja la contribución efectiva de cada característica dentro del contexto completo del modelo. Los resultados muestran una concentración importante de la capacidad predictiva en un grupo relativamente pequeño de variables.

**1. Variables Dominantes del Modelo**

Las variables más importantes fueron:

- `bathrooms_log`: 0.177
- `room_type_entire_home/apt`: 0.151
- `accommodates_log`: 0.131

Estas tres variables acumulan aproximadamente un 45.9% de toda la importancia del modelo.

Lo anterior indica que gran parte de la capacidad predictiva está explicada por el tamaño del alojamiento, la capacidad de hospedaje y el el tipo de habitación ofertada. Desde una perspectiva de negocio, este resultado es completamente razonable. Alojamientos más amplios, con mayor capacidad y que ofrecen la propiedad completa suelen presentar diferencias de precio significativas respecto a habitaciones privadas o compartidas.

**2. Importancia de la Ubicación**

Las variables geoespaciales mantienen una participación muy relevante:

- `dist_to_nearest_attraction_log`
- `commercial_within_radius`
- `attractions_within_radius_log`
- `neighbourhood_cleansed_freq`

En conjunto representan una fracción importante de la capacidad predictiva del modelo. Este resultado confirma una observación consistente a lo largo de todos los análisis realizados: la ubicación es uno de los factores estructurales más importantes para explicar el precio de los alojamientos. Además, estas variables continúan mostrando relevancia incluso después de haber sido evaluadas conjuntamente con información del alojamiento, del anfitrión y de las amenidades.

**3. Amenity Score vs Amenity Count**

Uno de los resultados más interesantes aparece en las variables agregadas de amenidades:

- `amenity_score`: 0.053
- `amenity_count`: 0.035

Ambas variables aportan información al modelo, pero `amenity_score` mantiene una importancia superior. Este resultado es consistente con los análisis previos:

- Mutual Information favorecía a `amenity_score`.
- VIF mostraba una fuerte relación entre ambas variables.
- La correlación entre ambas era elevada.

En consecuencia, existe evidencia sólida para considerar que `amenity_score` captura mejor la calidad y relevancia de las amenidades disponibles, mientras que `amenity_count` aporta información parcialmente redundante. Por esta razón, `amenity_count` continúa siendo uno de los principales candidatos a eliminación durante la selección final.

**4. Variables Derivadas del Host**

Las variables relacionadas con el anfitrión presentan una contribución moderada:

- `calculated_host_listings_count_private_rooms_log`
- `calculated_host_listings_count_entire_homes_log`
- `host_total_listings_segment_ordinal`
- `host_verifications_grouped_ordinal`
- `host_is_superhost`
- `has_review`

Aunque ninguna de ellas domina el modelo, varias mantienen una importancia estable. Esto sugiere que las características del host podrían contienen señal predictiva útil, pero secundaria respecto a factores como ubicación, tamaño y tipo de alojamiento.

**4. Amenidades Individuales**

Las amenidades seleccionadas muestran un comportamiento interesante. La única amenidad individual que alcanza una importancia considerable es `has_air_conditioning`. Mientras que otras amenidades previamente destacadas mediante correlación o Mutual Information presentan una contribución mucho menor:

- `has_tv`
- `has_elevator`
- `has_self_check_in`
- `has_coffee_maker`
- `has_outdoor_furniture`
-` has_free_parking`

Esto sugiere que gran parte de la información contenida en estas variables ya está siendo absorbida por otras características del modelo, especialmente por `amenity_score`. En otras palabras, las amenidades individuales parecen aportar información complementaria, pero no constituyen factores dominantes cuando se consideran conjuntamente con el resto de variables.

**5. Variables con Baja Contribución**

Algunas variables muestran una importancia extremadamente reducida:

- `property_group_other`
- `room_type_hotel_room`
- `property_group_unique/nature`
- `room_type_private_room`
- `property_group_guesthouse`
- `property_group_apartment`

Estos resultados indican que dichas variables apenas contribuyen al desempeño del modelo una vez consideradas las demás características. Por tanto, representan candidatas naturales para una posible simplificación del conjunto de variables.

**6. Comparación con Mutual Information**

Una de las conclusiones más valiosas surge al comparar Mutual Information con Feature Importance. Algunas variables que mostraban una señal individual importante reducen considerablemente su importancia cuando compiten con el resto de características, mientras que otras variables estructurales mantienen una relavancia elevada en todos los análisis realizados.Esto indica que estas últimas contienen información única y difícilmente reemplazable por otras variables del dataset.

Los resultados sugieren que la capacidad predictiva del modelo se concentra principalmente en cinco grandes grupos de variables:

1. Tamaño y capacidad del alojamiento.
2. Tipo de habitación y tipo de propiedad.
3. Ubicación y accesibilidad.
4. Nivel general de equipamiento (`amenity_score`).
5. Características y escala operativa del anfitrión.

Asimismo, el análisis aporta evidencia adicional para reconsiderar la permanencia de algunas variables altamente redundantes o de baja contribución, especialmente aquellas relacionadas con categorías poco frecuentes o amenidades individuales cuyo aporte marginal resulta limitado frente a variables agregadas como `amenity_score`.

En conjunto, los resultados obtenidos proporcionan una base sólida para realizar la selección final de variables que será utilizada durante la etapa de modelado.

### Selección final de variables

Tras completar el proceso de creación, transformación y evaluación de características, se realizó una selección final de variables con el objetivo de construir un conjunto de features que mantuviera un equilibrio entre capacidad predictiva, interpretabilidad y complejidad del modelo.

La selección no se basó en una única métrica, sino en la combinación de múltiples análisis realizados a lo largo del proceso de feature engineering:

- Correlación y redundancia entre variables.
- Análisis de multicolinealidad mediante Variance Inflation Factor (VIF).
- Relevancia individual mediante Mutual Information.
- Contribución conjunta mediante Feature Importance.
- Consideraciones de interpretabilidad y significado de negocio.

Este enfoque permitió identificar variables que aportan información única y relevante para la predicción del precio, así como detectar características redundantes o con baja contribución que podían ser eliminadas sin afectar significativamente el desempeño esperado del modelo. El resultado es un conjunto final de variables que concentra la mayor parte de la señal predictiva identificada durante el análisis exploratorio y la evaluación de características.

Es importante destacar que las técnicas de preprocesamiento implementadas durante este notebook tuvieron como propósito facilitar el análisis, evaluación y selección de variables. Por esta razón, las métricas y criterios utilizados para la selección final de features fueron calculados sobre el conjunto de datos completo, ya que esta etapa tuvo un carácter exploratorio y de diagnóstico, orientado a comprender el comportamiento de las variables y definir el conjunto final de features que será utilizado durante el modelado.

Como resultado de este proceso, el dataset final `df_model` será con su versión previa (`df_base_model`) al ajuste de transformaciones dependientes de los datos. De esta manera, operaciones como imputación de valores faltantes, codificación de variables categóricas, escalamiento y otras transformaciones que requieren aprender parámetros a partir de los datos serán ajustadas posteriormente utilizando únicamente el conjunto de entrenamiento. Este enfoque permite evitar data leakage y garantiza que el pipeline de preprocesamiento pueda reproducirse de forma consistente durante las etapas de validación, prueba e inferencia en producción.

In [77]:
# Selection of variables to model

df_model = df_base_model[[

    # Numeric features
    'calculated_host_listings_count_entire_homes',
    'calculated_host_listings_count_private_rooms',
    'dist_to_nearest_attraction', 'beds', 'amenity_score',
    'accommodates', 'bedrooms', 'bathrooms',
    'attractions_within_radius', 'commercial_within_radius',

    # Categorical features
    'room_type','neighbourhood_cleansed', 'property_group_room',
    'host_total_listings_segment', 'review_scores_mean_segment', 
    'minimum_nights_segment', 'host_verifications_grouped',

    # Boolean features
    'host_is_superhost','instant_bookable', 
    'has_tv', 'has_elevator', 'has_free_parking', 'has_coffee_maker', 
    'has_outdoor_furniture', 'has_air_conditioning','has_self_check_in', 'has_pool',

    # target
    'log_price'

]].copy()

df_model.info()

<class 'pandas.DataFrame'>
RangeIndex: 21450 entries, 0 to 21449
Data columns (total 28 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   calculated_host_listings_count_entire_homes   21450 non-null  int64  
 1   calculated_host_listings_count_private_rooms  21450 non-null  int64  
 2   dist_to_nearest_attraction                    21450 non-null  float64
 3   beds                                          21434 non-null  float64
 4   amenity_score                                 21450 non-null  float64
 5   accommodates                                  21450 non-null  int64  
 6   bedrooms                                      21353 non-null  float64
 7   bathrooms                                     21438 non-null  float64
 8   attractions_within_radius                     21450 non-null  int64  
 9   commercial_within_radius                      21450 non-null  int64  
 1

#### Variables eliminadas

- `amenity_count`

La variable `amenity_count` fue descartada debido a su alta redundancia con `amenity_score`. Durante el análisis de correlación se observó una correlación superior a 0.88 entre ambas variables, mientras que el análisis de VIF mostró que `amenity_score` concentraba gran parte de la multicolinealidad presente en el conjunto de datos.

Además, tanto Mutual Information como Feature Importance mostraron consistentemente un mejor desempeño para `amenity_score`, sugiriendo que esta variable captura de forma más efectiva la calidad y relevancia de las amenidades disponibles, mientras que `amenity_count` aporta información parcialmente redundante.


-  Variables `property_group_*`

Las variables derivadas mediante One-Hot Encoding de "property_group" fueron eliminadas de la selección final:

- `property_group_apartment`
- `property_group_guesthouse`
- `property_group_hotel`
- `property_group_house`
- `property_group_other`
- `property_group_unique/nature`

Esta decisión se tomó debido a que la variable codificada mediante Frequency Encoding (`property_group_room_freq`) mostró un desempeño superior en los análisis realizados. Particularmente, `property_group_room_freq` obtuvo uno de los valores más altos de Mutual Information dentro de todo el conjunto de variables y mantuvo una importancia relevante dentro del modelo Random Forest.

En contraste, la mayoría de las categorías individuales presentaron valores bajos tanto en Mutual Information como en Feature Importance, indicando una contribución limitada una vez considerada la información contenida en la variable codificada por frecuencia. Por ello, se optó por conservar la representación más compacta e informativa de la variable original.


- `has_review`

La variable `has_review` fue descartada debido a su limitada capacidad predictiva. Aunque inicialmente fue considerada durante el proceso de evaluación, presentó valores reducidos tanto en Mutual Information como en Feature Importance. Adicionalmente, al tratarse de una característica derivada de la existencia de reseñas, parte de la información que aporta ya se encuentra representada indirectamente por otras variables relacionadas con reputación y calidad del alojamiento, como `review_scores_mean_segment_ordinal`.

#### Conclusiones y próximos pasos

El dataset final (`df_model`) concentra las variables que demostraron aportar información relevante durante los análisis de correlación, multicolinealidad, Mutual Information, desempeño del modelo base e importancia de variables. Asimismo, se eliminaron características redundantes o de baja contribución con el objetivo de reducir la complejidad del modelo y mejorar su interpretabilidad.

Como resultado de este proceso, el dataset será almacenado para ser utilizado en la siguiente fase del proyecto, dedicada al desarrollo y evaluación de modelos de Machine Learning.

La siguiente etapa del proyecto estará enfocada en el modelado predictivo e incluirá:

1. División final de los datos en conjuntos de entrenamiento y prueba.
2. Entrenamiento de modelos base de referencia.
3. Entrenamiento y comparación de algoritmos más avanzados.
4. Optimización de hiperparámetros.
5. Evaluación de desempeño mediante métricas de regresión.
6. Comparación entre modelos candidatos.
7. Selección del modelo final.
8. Interpretación de resultados y preparación del modelo para el despliegue.

Con esto, queda concluida la fase de Feature Engineering, preparación de datos y selección de variables, estableciendo una base sólida para la construcción, evaluación y posterior despliegue de un modelo predictivo de precios de Airbnb. 

In [78]:
# Save df_model for the Modeling stage
df_model.to_csv("../data/processed/df_model.csv", index=False)